# Chapter 6 Systems of Linear Equations: Exercises

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asj252/linear-algebra-with-python/blob/main/en/EN_Chap6_Linear_System_Exercises.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/asj252/linear-algebra-with-python/HEAD?labpath=en%2FEN_Chap6_Linear_System_Exercises.ipynb)

Compiled from the original manuscript of Chapter 6 exercises provided by Professor Ping-Zen Ong: 44 top-level exercises in all, in the same order as in the manuscript. Every exercise comes with a hint and a complete solution that can be expanded; computational exercises are followed by a NumPy verification, while pure proof exercises have no accompanying program.

This part uses 0-based indexing. Inconsistent notation and ambiguities in the statements of the original manuscript are noted in the corresponding exercises, and computed results follow the original data.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True, linewidth=100)

def print_header(title):
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)

def print_step(step, desc):
    print(f"\n▶ Step {step}: {desc}")
    print("-" * 40)

def compare_print(label, actual, expected_desc):
    print(f"\n[{label}]")
    print(f"  Computed value: {actual}")
    print(f"  Expected value: {expected_desc}")

def check(label, actual, expected):
    compare_print(label, actual, expected)
    assert np.allclose(actual, expected), label
    print("  check passed")

def rref(A, tol=1e-10):
    """Gauss–Jordan elimination with partial pivoting; only for the small numerical examples of this chapter."""
    R = np.array(A, dtype=float, copy=True)
    row = 0
    for col in range(R.shape[1]):
        if row == R.shape[0]:
            break
        pivot = row + np.argmax(np.abs(R[row:, col]))
        if abs(R[pivot, col]) <= tol:
            continue
        R[[row, pivot]] = R[[pivot, row]]
        R[row] /= R[row, col]
        for i in range(R.shape[0]):
            if i != row:
                R[i] -= R[i, col] * R[row]
        row += 1
    R[np.abs(R) < tol] = 0
    return R

def lu_no_pivot(A):
    """Only for the square matrices in this book whose first n-1 pivots are nonzero."""
    U = np.array(A, dtype=float, copy=True)
    n = len(U)
    L = np.eye(n)
    for j in range(n - 1):
        assert abs(U[j, j]) > 1e-10, "zero pivot: this function does not exchange rows"
        for i in range(j + 1, n):
            L[i, j] = U[i, j] / U[j, j]
            U[i] -= L[i, j] * U[j]
    return L, U


### Row Operations, Reduced Forms, and Rank

::::{exercise} Row-Reduced Form
:label: exer-ch6-row-reduction

Let $\mathbf{A}=\begin{pmatrix}1&-2&3&1\\2&-4&8&6\\-1&2&-2&1\end{pmatrix}$.
1. Use elementary row operations to find its rref, writing out the steps in detail.
2. Identify the pivot positions and find $\operatorname{rank}(\mathbf{A})$.

**Hint.** First eliminate the entries below the pivot in column 0.

:::{solution} exer-ch6-row-reduction
:class: dropdown

**Step 1.** Perform $R_1\leftarrow R_1-2R_0$ and $R_2\leftarrow R_2+R_0$ in turn to obtain
$$\begin{pmatrix}1&-2&3&1\\0&0&2&4\\0&0&1&2\end{pmatrix}.$$
**Step 2.** Perform $R_1\leftarrow R_1/2$, $R_2\leftarrow R_2-R_1$, and $R_0\leftarrow R_0-3R_1$ to obtain
$$\underline{\operatorname{rref}(\mathbf{A})=\begin{pmatrix}1&-2&0&-5\\0&0&1&2\\0&0&0&0\end{pmatrix}}.$$
The pivots are the $1$s at positions $(0,0)$ and $(1,2)$, so $\underline{\operatorname{rank}(\mathbf{A})=2}$. Here $R_i$ denotes row $i$.
:::
::::

In [2]:
# ============================================================
print_header("Exercise 6-1 | Row-Reduced Form")
# ============================================================

# --- Step 1 ---
print_step(1, "Build the matrix and perform Gauss–Jordan elimination")
A = np.array([[1,-2,3,1],[2,-4,8,6],[-1,2,-2,1]])
check("rref(A)", rref(A), [[1,-2,0,-5],[0,0,1,2],[0,0,0,0]])
# --- Step 2 ---
print_step(2, "Check the rank")
check("rank(A)", np.linalg.matrix_rank(A), 2)

  Exercise 6-1 | Row-Reduced Form

▶ Step 1: Build the matrix and perform Gauss–Jordan elimination
----------------------------------------

[rref(A)]
  Computed value: [[ 1. -2.  0. -5.]
 [ 0.  0.  1.  2.]
 [ 0.  0.  0.  0.]]
  Expected value: [[1, -2, 0, -5], [0, 0, 1, 2], [0, 0, 0, 0]]
  check passed

▶ Step 2: Check the rank
----------------------------------------

[rank(A)]
  Computed value: 2
  Expected value: 2
  check passed


::::{exercise} Different Ways of Eliminating
:label: exer-ch6-two-echelon-paths

Let $\mathbf{B}=\begin{pmatrix}2&4&6\\1&3&5\end{pmatrix}$.
1. Multiply row 0 by $\frac{1}{2}$ and subtract row 1; call the result $\mathbf{E}_0$. Is it a ref?
2. Replace row 1 by "2 times row 1 minus row 0"; call the result $\mathbf{E}_1$. Is it a ref?
3. Compare the two refs, then find the rref of each and compare.

**Note on the original manuscript.** Method one does not say which row the result replaces; under the natural reading $\text{row}_1\leftarrow\frac12\text{row}_0-\text{row}_1$ a ref is obtained, and the statement holds. Method two does not specify which row is replaced either; here we stipulate that it replaces row 1.

**Hint.** A ref can change with the operations used, but the rref is unique.

:::{solution} exer-ch6-two-echelon-paths
:class: dropdown

**Step 1.** Method one gives $\mathbf{E}_0=\begin{pmatrix}2&4&6\\0&-1&-2\end{pmatrix}$; method two gives $\mathbf{E}_1=\begin{pmatrix}2&4&6\\0&2&4\end{pmatrix}$. Both are refs, but they are different.

**Step 2.** For $\mathbf{E}_0$, first perform $R_0\leftarrow R_0/2$ and $R_1\leftarrow-R_1$, and then $R_0\leftarrow R_0-2R_1$; for $\mathbf{E}_1$, first divide each of the two rows by $2$, and then perform the same elimination. Both give
$$\underline{\mathbf{R}_0=\mathbf{R}_1=\begin{pmatrix}1&0&-1\\0&1&2\end{pmatrix}}.$$
If the original sentence is read as replacing row 0, we first obtain $\begin{pmatrix}0&2&4\\1&3&5\end{pmatrix}$, which is not a ref, but continuing the elimination still gives the rref above.
:::
::::

In [3]:
# ============================================================
print_header("Exercise 6-2 | Different Ways of Eliminating")
# ============================================================

# --- Step 1 ---
print_step(1, "Compare three readings of the operations")
expected = [[1,0,-1],[0,1,2]]
for B in ([[2,4,6],[0,-1,-2]], [[2,4,6],[0,2,4]], [[0,2,4],[1,3,5]]):
    check("rref", rref(B), expected)

  Exercise 6-2 | Different Ways of Eliminating

▶ Step 1: Compare three readings of the operations
----------------------------------------

[rref]
  Computed value: [[ 1.  0. -1.]
 [ 0.  1.  2.]]
  Expected value: [[1, 0, -1], [0, 1, 2]]
  check passed

[rref]
  Computed value: [[ 1.  0. -1.]
 [ 0.  1.  2.]]
  Expected value: [[1, 0, -1], [0, 1, 2]]
  check passed

[rref]
  Computed value: [[ 1.  0. -1.]
 [ 0.  1.  2.]]
  Expected value: [[1, 0, -1], [0, 1, 2]]
  check passed


::::{exercise} Recovering the Matrix
:label: exer-ch6-recover-columns

Suppose $\operatorname{rref}(\mathbf{A})=\begin{pmatrix}1&3&0&2\\0&0&1&-1\\0&0&0&0\end{pmatrix}$, and columns 0 and 2 are $(1,2,1)^\top$ and $(2,5,3)^\top$, respectively.
1. Recover $\mathbf{A}$.
2. Find a basis of the row space.
3. Find a basis of the column space.

**Hint.** Invertible row operations preserve the linear relations among the columns.

:::{solution} exer-ch6-recover-columns
:class: dropdown

**Step 1.** The rref shows that column 1 is $3$ times column 0, and column 3 is $2$ times column 0 minus column 2. Therefore
$$\underline{\mathbf{A}=\begin{pmatrix}1&3&2&0\\2&6&5&-1\\1&3&3&-1\end{pmatrix}}.$$
**Step 2.** Row operations preserve the row space, so a basis of the row space is $\underline{\{(1,3,0,2),(0,0,1,-1)\}}$. For the column space, take the pivot columns of the original matrix; a basis is $\underline{\{(1,2,1)^\top,(2,5,3)^\top\}}$. The pivot columns of the rref cannot be used directly as a basis of the original column space.
:::
::::

In [4]:
# ============================================================
print_header("Exercise 6-3 | Recovering the Matrix")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the rref of the recovered matrix")
A = np.array([[1,3,2,0],[2,6,5,-1],[1,3,3,-1]])
check("rref(A)", rref(A), [[1,3,0,2],[0,0,1,-1],[0,0,0,0]])
check("column relation", A[:,3], 2*A[:,0]-A[:,2])
check("rank of the two basis columns", np.linalg.matrix_rank(A[:,[0,2]]), 2)

  Exercise 6-3 | Recovering the Matrix

▶ Step 1: Check the rref of the recovered matrix
----------------------------------------

[rref(A)]
  Computed value: [[ 1.  3.  0.  2.]
 [ 0.  0.  1. -1.]
 [ 0.  0.  0.  0.]]
  Expected value: [[1, 3, 0, 2], [0, 0, 1, -1], [0, 0, 0, 0]]
  check passed

[column relation]
  Computed value: [ 0 -1 -1]
  Expected value: [ 0 -1 -1]
  check passed

[rank of the two basis columns]
  Computed value: 2
  Expected value: 2
  check passed


::::{exercise} Undoing Row Operations
:label: exer-ch6-undo-elementary-operations

Apply the following to a $3\times3$ matrix $\mathbf{M}$ in turn: add $-2$ times row 0 to row 1, interchange rows 1 and 2, and divide row 2 by $3$.
1. Write down the corresponding elementary matrices $\mathbf{E}_0,\mathbf{E}_1,\mathbf{E}_2$.
2. Given that the result is $\boldsymbol{\Gamma}=\begin{pmatrix}3&1&1\\0&2&1\\0&0&1\end{pmatrix}$, recover $\mathbf{M}$ from $\mathbf{E}_2\mathbf{E}_1\mathbf{E}_0\mathbf{M}=\boldsymbol{\Gamma}$.

**Hint.** Undo the operations one by one in reverse order.

:::{solution} exer-ch6-undo-elementary-operations
:class: dropdown

**Step 1.** The three elementary matrices are
$$\mathbf{E}_0=\begin{pmatrix}1&0&0\\-2&1&0\\0&0&1\end{pmatrix}, 
\mathbf{E}_1=\begin{pmatrix}1&0&0\\0&0&1\\0&1&0\end{pmatrix}, 
\mathbf{E}_2=\operatorname{diag}(1,1,1/3).$$
**Step 2.** Starting from the result, first multiply row 2 by $3$, then interchange rows 1 and 2, and finally add $2$ times row 0 to row 1; hence
$$\underline{\mathbf{M}=\mathbf{E}_0^{-1}\mathbf{E}_1^{-1}\mathbf{E}_2^{-1}\boldsymbol{\Gamma}=\begin{pmatrix}3&1&1\\6&2&5\\0&2&1\end{pmatrix}}.$$
:::
::::

In [5]:
# ============================================================
print_header("Exercise 6-4 | Undoing Row Operations")
# ============================================================

# --- Step 1 ---
print_step(1, "Redo the three row operations after recovering M")
E0 = np.array([[1,0,0],[-2,1,0],[0,0,1]])
E1 = np.eye(3)[[0,2,1]]
E2 = np.diag([1,1,1/3])
M = np.array([[3,1,1],[6,2,5],[0,2,1]])
check("E2 E1 E0 M", E2@E1@E0@M, [[3,1,1],[0,2,1],[0,0,1]])

  Exercise 6-4 | Undoing Row Operations

▶ Step 1: Redo the three row operations after recovering M
----------------------------------------

[E2 E1 E0 M]
  Computed value: [[3. 1. 1.]
 [0. 2. 1.]
 [0. 0. 1.]]
  Expected value: [[3, 1, 1], [0, 2, 1], [0, 0, 1]]
  check passed


::::{exercise} A Parameter and the Rank
:label: exer-ch6-rank-two-rows

Let $\mathbf{A}=\begin{pmatrix}1&-2&1\\-3&6&\lambda\end{pmatrix}$.
1. Use row operations to find a ref.
2. When does the column space have dimension $1$? What is the rank in the other cases?

**Hint.** Watch whether row 1 vanishes.

:::{solution} exer-ch6-rank-two-rows
:class: dropdown

**Step 1.** Performing $R_1\leftarrow R_1+3R_0$ gives the ref
$$\begin{pmatrix}1&-2&1\\0&0&\lambda+3\end{pmatrix}.$$
**Step 2.** When $\lambda=-3$ only one pivot remains; in the other cases there are two pivots. Therefore
$$\underline{\dim\operatorname{Col}(\mathbf{A})=\operatorname{rank}(\mathbf{A})=\begin{cases}1,&\lambda=-3,\\2,&\lambda\ne-3.\end{cases}}$$
:::
::::

In [6]:
# ============================================================
print_header("Exercise 6-5 | A Parameter and the Rank")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the degenerate value and generic values")
for lam in [-3,-2,0,4]:
    A = [[1,-2,1],[-3,6,lam]]
    check(f"λ={lam}: rank", np.linalg.matrix_rank(A), 1 if lam == -3 else 2)

  Exercise 6-5 | A Parameter and the Rank

▶ Step 1: Check the degenerate value and generic values
----------------------------------------

[λ=-3: rank]
  Computed value: 1
  Expected value: 1
  check passed

[λ=-2: rank]
  Computed value: 2
  Expected value: 2
  check passed

[λ=0: rank]
  Computed value: 2
  Expected value: 2
  check passed

[λ=4: rank]
  Computed value: 2
  Expected value: 2
  check passed


::::{exercise} Classifying Reduced Forms
:label: exer-ch6-all-rref-three

1. List all $3\times3$ rref matrices.
2. If $\boldsymbol{\Gamma}$ is such a matrix of rank $r$, prove that $\operatorname{Col}(\boldsymbol{\Gamma})=\operatorname{span}(\mathbf{e}_0,\ldots,\mathbf{e}_{r-1})$; when $r=0$, read this as the zero subspace.
3. Prove that sending an rref matrix to its row space gives a bijection between these matrices and all the subspaces of $\mathbb{R}^3$.

**Hint.** Classify by the rank and by the columns containing the pivots, and use the uniqueness of the rref. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-all-rref-three
:class: dropdown

1. Rank $0$ gives only $\mathbf{0}$, and rank $3$ gives only $\mathbf{I}_3$. All the forms of rank $1$ are
   $$\begin{pmatrix}1&a&b\\0&0&0\\0&0&0\end{pmatrix}, 
   \begin{pmatrix}0&1&a\\0&0&0\\0&0&0\end{pmatrix}, 
   \begin{pmatrix}0&0&1\\0&0&0\\0&0&0\end{pmatrix}.$$
   All the forms of rank $2$ are
   $$\begin{pmatrix}1&0&a\\0&1&b\\0&0&0\end{pmatrix}, 
   \begin{pmatrix}1&a&0\\0&0&1\\0&0&0\end{pmatrix}, 
   \begin{pmatrix}0&1&0\\0&0&1\\0&0&0\end{pmatrix},$$
   where $a,b\in\mathbb{R}$ are arbitrary. These exhaust all choices of the pivot-column positions.
2. Everything below the first $r$ nonzero rows is zero, so every column lies in $\operatorname{span}(\mathbf{e}_0,\ldots,\mathbf{e}_{r-1})$; the pivot columns are exactly these standard basis vectors, so the reverse inclusion also holds.
3. A basis of any subspace $W$ can be arranged as the nonzero rows of a matrix, padded with zero rows to size $3\times3$; its rref still has $W$ as its row space, so the correspondence is surjective. If two rrefs have the same row space, their nonzero rows are two bases of the same space, so the matrices are row equivalent; by the uniqueness of the rref proved in the next group of exercises, they are equal, so the correspondence is also injective. $\square$
:::
::::

::::{exercise} Row Equivalence
:label: exer-ch6-row-equivalence-kernel

Let $\mathbf{A},\mathbf{B}$ be $m\times n$ matrices such that $\mathbf{B}$ is obtained from $\mathbf{A}$ by finitely many elementary row operations.
1. Prove that there exists an invertible $m\times m$ matrix $\mathbf{M}$ with $\mathbf{B}=\mathbf{M}\mathbf{A}$.
2. Are the kernels equal? Are the images equal? Give a proof or a counterexample.

**Hint.** Every elementary matrix is invertible, but left multiplication changes the position of the image within the ambient space. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-row-equivalence-kernel
:class: dropdown

1. Let the elementary matrices applied on the left in turn be $\mathbf{E}_0,\ldots,\mathbf{E}_{k-1}$; then $\mathbf{M}=\mathbf{E}_{k-1}\cdots\mathbf{E}_0$ is invertible and $\mathbf{B}=\mathbf{M}\mathbf{A}$; if there are no operations, take $\mathbf{M}=\mathbf{I}_m$.
2. Since $\mathbf{M}$ is invertible, $\mathbf{B}\mathbf{x}=\mathbf{0}$ is equivalent to $\mathbf{A}\mathbf{x}=\mathbf{0}$, so the kernels are the same. The images satisfy only $\operatorname{Im}(\mathbf{B})=\mathbf{M}\operatorname{Im}(\mathbf{A})$ and need not be equal. For example, for $\mathbf{A}=(1,0)^\top$, interchanging the two rows gives $\mathbf{B}=(0,1)^\top$; the images are two different coordinate axes. $\square$
:::
::::

::::{exercise} Uniqueness of the Reduced Form
:label: exer-ch6-rref-uniqueness

Prove that the rref of any matrix is unique.

**Hint.** Use the row space to determine the pivot positions, then compare rows with the same pivot coordinates. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-rref-uniqueness
:class: dropdown

1. Elementary row operations preserve the row space. Let $W$ be the row space of the original matrix, and let $\mathbf{R}$ be an rref whose nonzero rows are $\mathbf{r}_0,\ldots,\mathbf{r}_{r-1}$, with pivot positions $p_0<\cdots<p_{r-1}$.
2. Write any nonzero vector $\mathbf{w}\in W$ as $\sum_{i=0}^{r-1}c_i\mathbf{r}_i$. If $c_k$ is the first nonzero coefficient, then the first nonzero component of $\mathbf{w}$ lies exactly at $p_k$. Hence the set of all possible positions of a first nonzero component is exactly $\{p_0,\ldots,p_{r-1}\}$, which is completely determined by $W$.
3. If $\mathbf{S}$ is another rref, it has the same pivot positions. For each $i$, the difference between the $i$-th nonzero rows of the two matrices lies in $W$, and all its components at the pivot positions are zero. But point 2 shows that a nonzero vector of $W$ must be nonzero at some pivot position, so this difference must be zero. The two matrices have the same nonzero rows and the same zero rows at the bottom, so $\mathbf{R}=\mathbf{S}$. $\square$
:::
::::

::::{exercise} Reduced Form with a Parameter
:label: exer-ch6-parameter-rref-three

Let $\mathbf{A}_\lambda=\begin{pmatrix}1&1&\lambda\\1&\lambda&1\\\lambda&1&1\end{pmatrix}$.
1. For every $\lambda\in\mathbb{R}$, use row operations to find the rref.
2. Classify the rank completely.

**Hint.** First separate out $\lambda=1$, then look at the last pivot.

:::{solution} exer-ch6-parameter-rref-three
:class: dropdown

**Step 1.** Performing $R_1\leftarrow R_1-R_0$ and $R_2\leftarrow R_2-\lambda R_0$ gives
$$\begin{pmatrix}1&1&\lambda\\0&\lambda-1&1-\lambda\\0&1-\lambda&1-\lambda^2\end{pmatrix}.$$
If $\lambda=1$, only the first row $(1,1,1)$ remains, and the rank is $1$.

**Step 2.** If $\lambda\ne1$, divide row 1 by $\lambda-1$, then perform $R_2\leftarrow R_2+(\lambda-1)R_1$; the last pivot is $-(\lambda-1)(\lambda+2)$. Hence
$$\underline{\operatorname{rref}(\mathbf{A}_\lambda)=\begin{cases}
\begin{pmatrix}1&1&1\\0&0&0\\0&0&0\end{pmatrix},&\lambda=1,\\
\begin{pmatrix}1&0&-1\\0&1&-1\\0&0&0\end{pmatrix},&\lambda=-2,\\
\mathbf{I}_3,&\lambda\notin\{1,-2\}.
\end{cases}}$$
The corresponding ranks are $\underline{1,2,3}$. In the nondegenerate case, eliminating upward from the last row gives the identity matrix.
:::
::::

In [7]:
# ============================================================
print_header("Exercise 6-9 | Reduced Form with a Parameter")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the two degenerate values and generic values separately")
for lam in [1,-2,0,2,-3]:
    A = np.array([[1,1,lam],[1,lam,1],[lam,1,1]])
    expected = ([[1,1,1],[0,0,0],[0,0,0]] if lam == 1 else
                [[1,0,-1],[0,1,-1],[0,0,0]] if lam == -2 else np.eye(3))
    check(f"λ={lam}: rref", rref(A), expected)

  Exercise 6-9 | Reduced Form with a Parameter

▶ Step 1: Check the two degenerate values and generic values separately
----------------------------------------

[λ=1: rref]
  Computed value: [[1. 1. 1.]
 [0. 0. 0.]
 [0. 0. 0.]]
  Expected value: [[1, 1, 1], [0, 0, 0], [0, 0, 0]]
  check passed

[λ=-2: rref]
  Computed value: [[ 1.  0. -1.]
 [ 0.  1. -1.]
 [ 0.  0.  0.]]
  Expected value: [[1, 0, -1], [0, 1, -1], [0, 0, 0]]
  check passed

[λ=0: rref]
  Computed value: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
  Expected value: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
  check passed

[λ=2: rref]
  Computed value: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
  Expected value: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
  check passed

[λ=-3: rref]
  Computed value: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
  Expected value: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
  check passed


::::{exercise} Rank and Degrees of Freedom
:label: exer-ch6-rank-nullity-six

A linear mapping $T:\mathbb{R}^6\to\mathbb{R}^4$ satisfies $\dim\ker T=3$, and its representing matrix is the $4\times6$ matrix $\mathbf{A}$.
1. How many pivots and free variables does the rref have?
2. Are there zero rows? How many, and where are they?

**Hint.** Use the fact that the rank plus the dimension of the kernel equals the dimension of the domain. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-rank-nullity-six
:class: dropdown

1. The rank–nullity theorem gives $\operatorname{rank}(\mathbf{A})=6-3=3$, so there are $3$ pivots and $6-3=3$ free variables.
2. The rref has $4$ rows, $3$ of which are nonzero, so there is exactly $1$ zero row; by the definition of the rref it must be at the bottom, that is, row $3$. $\square$
:::
::::

### Homogeneous Systems of Linear Equations

::::{exercise} Solving Homogeneous Systems
:label: exer-ch6-three-homogeneous-systems

Use Gaussian elimination to find the complete solution of each of the following three homogeneous systems:
1. $x+2y-z=0$, $3x+6y-2z=0$, $-2x-4y+7z=0$.
2. $x-y+2z=0$, $2x+y-z=0$, $-x+2y+z=0$, $x+4y+3z=0$.
3. $2z=0$, $x+2y+2z=0$, $3x+6y+5z=0$.

**Hint.** A homogeneous system always has at least the zero solution; the number of pivots determines the degrees of freedom.

:::{solution} exer-ch6-three-homogeneous-systems
:class: dropdown

**Step 1.** For system 1, after elimination with row 0 the two lower equations are $z=0$ and $5z=0$, so further elimination gives the rref
$$\begin{pmatrix}1&2&0\\0&0&1\\0&0&0\end{pmatrix}.$$
Hence $\underline{(x,y,z)^\top=t(-2,1,0)^\top}$, $t\in\mathbb{R}$.

**Step 2.** For system 2, after eliminating column 0 the lower three rows are $(0,3,-5)$, $(0,1,3)$, and $(0,5,1)$. After elimination with $(0,1,3)$, a nonzero pivot appears in column 2 (for example $-14$), so the rref is $\mathbf{I}_3$ on top with one zero row at the bottom; hence $\underline{(x,y,z)=(0,0,0)}$.

**Step 3.** For system 3, first interchange the first two rows, then use row 0 to eliminate in row 2, which gives $(0,0,-1)$; row 1 gives $z=0$, and what remains is $x+2y=0$. Hence $\underline{(x,y,z)^\top=t(-2,1,0)^\top}$.
:::
::::

In [8]:
# ============================================================
print_header("Exercise 6-11 | Solving Homogeneous Systems")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the three rrefs and the kernel vectors")
matrices=[[[1,2,-1],[3,6,-2],[-2,-4,7]], [[1,-1,2],[2,1,-1],[-1,2,1],[1,4,3]], [[0,0,2],[1,2,2],[3,6,5]]]
for i,A in enumerate(matrices):
    expected=np.vstack([np.eye(3),np.zeros((1,3))]) if i==1 else [[1,2,0],[0,0,1],[0,0,0]]
    check(f"system {i+1}: rref", rref(A), expected)
    if i != 1:check("kernel vector", np.array(A)@np.array([-2,1,0]), np.zeros(3))

  Exercise 6-11 | Solving Homogeneous Systems

▶ Step 1: Check the three rrefs and the kernel vectors
----------------------------------------

[system 1: rref]
  Computed value: [[1. 2. 0.]
 [0. 0. 1.]
 [0. 0. 0.]]
  Expected value: [[1, 2, 0], [0, 0, 1], [0, 0, 0]]
  check passed

[kernel vector]
  Computed value: [0 0 0]
  Expected value: [0. 0. 0.]
  check passed

[system 2: rref]
  Computed value: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 0.]]
  Expected value: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 0.]]
  check passed

[system 3: rref]
  Computed value: [[1. 2. 0.]
 [0. 0. 1.]
 [0. 0. 0.]]
  Expected value: [[1, 2, 0], [0, 0, 1], [0, 0, 0]]
  check passed

[kernel vector]
  Computed value: [0 0 0]
  Expected value: [0. 0. 0.]
  check passed


::::{exercise} Kernel in Four Unknowns
:label: exer-ch6-four-variable-kernel

Consider $x_0+2x_1-x_2+3x_3=0$, $2x_0+4x_1-x_2+8x_3=0$, $-x_0-2x_1+2x_2-x_3=0$.
1. Write the system as $\mathbf{A}\mathbf{x}=\mathbf{0}$.
2. Find the rref $\boldsymbol{\Gamma}$ of the coefficient matrix.
3. Identify the pivot variables and the free variables.
4. Find the general solution, a basis of the kernel, and its dimension.

**Hint.** Keep $x_1,x_3$ as free parameters.

:::{solution} exer-ch6-four-variable-kernel
:class: dropdown

**Step 1.** $\mathbf{x}=(x_0,x_1,x_2,x_3)^\top$, and
$$\mathbf{A}=\begin{pmatrix}1&2&-1&3\\2&4&-1&8\\-1&-2&2&-1\end{pmatrix}.$$
After $R_1\leftarrow R_1-2R_0$ and $R_2\leftarrow R_2+R_0$, the two lower rows are both $(0,0,1,2)$; eliminate row 2, then perform $R_0\leftarrow R_0+R_1$ to obtain
$$\boldsymbol{\Gamma}=\begin{pmatrix}1&2&0&5\\0&0&1&2\\0&0&0&0\end{pmatrix}.$$
**Step 2.** The pivot variables are $x_0,x_2$, and the free variables are $x_1=s,x_3=t$; therefore
$$\underline{\mathbf{x}=s(-2,1,0,0)^\top+t(-5,0,-2,1)^\top},  s,t\in\mathbb{R}.$$
In the free coordinates the two vectors are $(1,0)$ and $(0,1)$, respectively, so they are linearly independent and form a basis of the kernel; $\underline{\dim\ker\mathbf{A}=2}$.
:::
::::

In [9]:
# ============================================================
print_header("Exercise 6-12 | Kernel in Four Unknowns")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the rref")
A=np.array([[1,2,-1,3],[2,4,-1,8],[-1,-2,2,-1]])
check("rref(A)", rref(A), [[1,2,0,5],[0,0,1,2],[0,0,0,0]])
# --- Step 2 ---
print_step(2, "Check the basis of the kernel")
N=np.array([[-2,-5],[1,0],[0,-2],[0,1]])
check("AN", A@N, np.zeros((3,2)))
check("rank(N)", np.linalg.matrix_rank(N), 2)

  Exercise 6-12 | Kernel in Four Unknowns

▶ Step 1: Check the rref
----------------------------------------

[rref(A)]
  Computed value: [[1. 2. 0. 5.]
 [0. 0. 1. 2.]
 [0. 0. 0. 0.]]
  Expected value: [[1, 2, 0, 5], [0, 0, 1, 2], [0, 0, 0, 0]]
  check passed

▶ Step 2: Check the basis of the kernel
----------------------------------------

[AN]
  Computed value: [[0 0]
 [0 0]
 [0 0]]
  Expected value: [[0. 0.]
 [0. 0.]
 [0. 0.]]
  check passed

[rank(N)]
  Computed value: 2
  Expected value: 2
  check passed


::::{exercise} Only the Zero Solution
:label: exer-ch6-full-rank-homogeneous

Let $\mathbf{B}=\begin{pmatrix}1&1&2\\1&2&5\\2&3&8\end{pmatrix}$, and consider $\mathbf{B}(x,y,z)^\top=\mathbf{0}$.
1. Use row operations to find a ref.
2. Is there a nonzero solution? Explain.

**Hint.** Is the last pivot zero?

:::{solution} exer-ch6-full-rank-homogeneous
:class: dropdown

**Step 1.** Performing $R_1\leftarrow R_1-R_0$, $R_2\leftarrow R_2-2R_0$, and $R_2\leftarrow R_2-R_1$ gives
$$\underline{\mathbf{U}=\begin{pmatrix}1&1&2\\0&1&3\\0&0&1\end{pmatrix}}.$$
**Step 2.** Back substitution gives $z=0,y=0,x=0$. All three unknowns have pivots, so $\underline{\ker\mathbf{B}=\{\mathbf{0}\}}$, and there is no nonzero solution.
:::
::::

In [10]:
# ============================================================
print_header("Exercise 6-13 | Only the Zero Solution")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the rank and the reduced form")
B=np.array([[1,1,2],[1,2,5],[2,3,8]])
check("rref(B)", rref(B), np.eye(3))
check("rank(B)", np.linalg.matrix_rank(B), 3)

  Exercise 6-13 | Only the Zero Solution

▶ Step 1: Check the rank and the reduced form
----------------------------------------

[rref(B)]
  Computed value: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
  Expected value: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
  check passed

[rank(B)]
  Computed value: 3
  Expected value: 3
  check passed


::::{exercise} Homogeneous Solutions with a Parameter
:label: exer-ch6-parameter-homogeneous

Consider $x+y+\lambda z=0$, $x+\lambda y+z=0$, $\lambda x+y+z=0$, where $\lambda\in\mathbb{R}$.
1. Use row operations to determine when there is only the zero solution.
2. When are there nonzero solutions? Find the dimension of the solution space and the general solution.

**Hint.** Use the three rrefs of the parameter matrix found earlier.

:::{solution} exer-ch6-parameter-homogeneous
:class: dropdown

**Step 1.** First subtract row 0 from row 1 and $\lambda$ times row 0 from row 2. When $\lambda\ne1$, divide row 1 by $\lambda-1$ and eliminate the component below it; the last pivot is $-(\lambda-1)(\lambda+2)$. So for $\underline{\lambda\notin\{1,-2\}}$ the matrix has full rank, there is only the zero solution, and the kernel has dimension $0$.

**Step 2.** When $\lambda=1$, only $x+y+z=0$ remains, so $\underline{(x,y,z)^\top=s(-1,1,0)^\top+t(-1,0,1)^\top}$, of dimension $2$. When $\lambda=-2$, the rref gives $x=z,y=z$, so $\underline{(x,y,z)^\top=t(1,1,1)^\top}$, of dimension $1$.
:::
::::

In [11]:
# ============================================================
print_header("Exercise 6-14 | Homogeneous Solutions with a Parameter")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the kernels and their dimensions in the degenerate cases")
for lam,N in [(1,np.array([[-1,-1],[1,0],[0,1]])),(-2,np.ones((3,1)))]:
    A=np.array([[1,1,lam],[1,lam,1],[lam,1,1]])
    check("AN", A@N, np.zeros((3,N.shape[1])))
    check("nullity(A)", 3-np.linalg.matrix_rank(A), N.shape[1])
check("only the zero solution when λ=0", np.linalg.matrix_rank([[1,1,0],[1,0,1],[0,1,1]]), 3)

  Exercise 6-14 | Homogeneous Solutions with a Parameter

▶ Step 1: Check the kernels and their dimensions in the degenerate cases
----------------------------------------

[AN]
  Computed value: [[0 0]
 [0 0]
 [0 0]]
  Expected value: [[0. 0.]
 [0. 0.]
 [0. 0.]]
  check passed

[nullity(A)]
  Computed value: 2
  Expected value: 2
  check passed

[AN]
  Computed value: [[0.]
 [0.]
 [0.]]
  Expected value: [[0.]
 [0.]
 [0.]]
  check passed

[nullity(A)]
  Computed value: 1
  Expected value: 1
  check passed

[only the zero solution when λ=0]
  Computed value: 3
  Expected value: 3
  check passed


::::{exercise} Solving from the Row Space
:label: exer-ch6-kernel-from-row-space

The row space of a real matrix $\mathbf{A}\in\mathbb{R}^{3\times3}$ is spanned by $(1,-1,2)$ and $(2,1,1)$.
1. Write down a system of equations with the same solutions as $\mathbf{A}\mathbf{x}=\mathbf{0}$, and explain why.
2. Find the general solution and determine its geometric shape.

**Hint.** A vector must be sent to zero by every row vector in the row space.

:::{solution} exer-ch6-kernel-from-row-space
:class: dropdown

**Step 1.** We may instead solve $x-y+2z=0$, $2x+y+z=0$. Every row of the original matrix is a linear combination of these two rows, and these two rows also lie in the original row space; hence each equation on either side follows from those on the other side by linear combination, and the solution sets are the same.

**Step 2.** Adding gives $3x+3z=0$, so $x=-z$, and substituting gives $y=z$. Therefore
$$\underline{\mathbf{x}=t(-1,1,1)^\top},  t\in\mathbb{R}.$$
The solution set is a line through the origin, of dimension $1$.
:::
::::

In [12]:
# ============================================================
print_header("Exercise 6-15 | Solving from the Row Space")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the kernel of the two equivalent equations")
B=np.array([[1,-1,2],[2,1,1]])
check("B(-1,1,1)^T", B@np.array([-1,1,1]), np.zeros(2))
check("rank(B)", np.linalg.matrix_rank(B), 2)

  Exercise 6-15 | Solving from the Row Space

▶ Step 1: Check the kernel of the two equivalent equations
----------------------------------------

[B(-1,1,1)^T]
  Computed value: [0 0]
  Expected value: [0. 0.]
  check passed

[rank(B)]
  Computed value: 2
  Expected value: 2
  check passed


::::{exercise} Constraints from a Nonzero Solution
:label: exer-ch6-rank-bound-nonzero-kernel

A real matrix $\mathbf{A}$ is $4\times3$, and $\mathbf{A}\mathbf{x}=\mathbf{0}$ is known to have a nonzero solution.
1. What are the largest and the smallest possible ranks?
2. Must the original system or the ref of its coefficient matrix have a zero row? Explain the difference between the two.

**Hint.** A nonzero kernel vector gives nullity at least $1$, but it does not force the original matrix to have a zero row. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-rank-bound-nonzero-kernel
:class: dropdown

1. A nonzero solution means $\dim\ker\mathbf{A}\ge1$, so the rank is at most $3-1=2$, and at least $0$. Both bounds are attained: the zero matrix attains $0$; the matrix with $\mathbf{e}_0^\top,\mathbf{e}_1^\top$ as its first two rows and zero rows elsewhere attains $2$.
2. The ref must have $4-r\ge2$ zero rows at the bottom, which means that after elimination there are redundant equations. But the original matrix need not have a zero row: for example, the matrix whose four rows all equal $(1,0,0)$ has a nonzero kernel vector, while every one of the original equations is nonzero. $\square$
:::
::::

### Nonhomogeneous Systems

::::{exercise} Solving Nonhomogeneous Systems
:label: exer-ch6-three-inhomogeneous-systems

Use Gaussian elimination to find the complete solution of each of the following three systems; if there are infinitely many solutions, describe the geometric relation to the solution set of the corresponding homogeneous system.
1. $x+2y-z=1$, $3x+6y-2z=0$, $-2x-4y+z=0$.
2. $x-y+2z=4$, $2x+y-z=0$, $-x+2y+z=-2$, $x+4y+3z=0$.
3. $2z=2$, $x+2y+2z=1$, $3x+6y+5z=2$.

**Note on the original manuscript.** The missing variable in equation 2 of system 1 has been fixed; but the new coefficient is $z$, which differs from the $7z$ in system 1 of {ref}`exer-ch6-three-homogeneous-systems`, so the statement's "reference" to the coefficients of the homogeneous version no longer matches. With either $z$ or $7z$ there is no solution, which is still consistent with the exercise asking for the complete solution only when a solution exists.

**Hint.** An augmented matrix with a row $[0 0 0\mid c]$, $c\ne0$, means there is no solution.

:::{solution} exer-ch6-three-inhomogeneous-systems
:class: dropdown

**Step 1.** For system 1, let $s=x+2y$. Subtracting $3$ times equation 0 from equation 1 gives $z=-3$, and then the first equation gives $s=-2$; but the left side of the last equation is $-2s+z=4-3=1\ne0$, so there is $\underline{\text{no solution}}$. In the augmented matrix, performing $R_1\leftarrow R_1-3R_0$, $R_2\leftarrow R_2+2R_0$, and $R_2\leftarrow R_2+R_1$ in turn makes the last row $[0,0,0\mid-1]$; so the coefficient matrix has rank $2$ and the augmented matrix has rank $3$. If, following the coefficients of the homogeneous version, the equation is read instead as $-2x-4y+7z=0$, substitution gives $4-21=-17\ne0$, and there is still no solution.

**Step 2.** For system 2, after elimination the first three equations give $3y-5z=-8$ and $y+3z=2$, so $z=1,y=-1,x=1$; equation 3 is also satisfied: $1-4+3=0$. Three pivots guarantee uniqueness, so $\underline{(x,y,z)=(1,-1,1)}$.

**Step 3.** For system 3, the first equation gives $z=1$, equation 1 becomes $x+2y=-1$, and equation 2 holds automatically after substitution. Therefore
$$\underline{(x,y,z)^\top=(-1,0,1)^\top+t(-2,1,0)^\top},  t\in\mathbb{R}.$$
Its solution set is the affine line obtained by translating the homogeneous solution line $\operatorname{span}((-2,1,0)^\top)$ by $(-1,0,1)^\top$.
:::
::::

In [13]:
# ============================================================
print_header("Exercise 6-17 | Solving Nonhomogeneous Systems")
# ============================================================

# --- Step 1 ---
print_step(1, "Check that both the new z and the 7z reading from the homogeneous version have no solution")
for A,b in [(np.array([[1,2,-1],[3,6,-2],[-2,-4,1]]),[1,0,0]),
            (np.array([[1,2,-1],[3,6,-2],[-2,-4,7]]),[1,0,0])]:
    check("rank(A)", np.linalg.matrix_rank(A), 2)
    check("rank([A | b])", np.linalg.matrix_rank(np.column_stack([A,b])), 3)
# --- Step 2 ---
print_step(2, "Verify the unique solution and the affine family of solutions")
A=np.array([[1,-1,2],[2,1,-1],[-1,2,1],[1,4,3]])
check("system 2 substituted", A@np.array([1,-1,1]), [4,0,-2,0])
A=np.array([[0,0,2],[1,2,2],[3,6,5]])
check("system 3 particular solution", A@np.array([-1,0,1]), [2,1,2])
check("system 3 homogeneous direction", A@np.array([-2,1,0]), [0,0,0])

  Exercise 6-17 | Solving Nonhomogeneous Systems

▶ Step 1: Check that both the new z and the 7z reading from the homogeneous version have no solution
----------------------------------------

[rank(A)]
  Computed value: 2
  Expected value: 2
  check passed

[rank([A | b])]
  Computed value: 3
  Expected value: 3
  check passed

[rank(A)]
  Computed value: 2
  Expected value: 2
  check passed

[rank([A | b])]
  Computed value: 3
  Expected value: 3
  check passed

▶ Step 2: Verify the unique solution and the affine family of solutions
----------------------------------------

[system 2 substituted]
  Computed value: [ 4  0 -2  0]
  Expected value: [4, 0, -2, 0]
  check passed

[system 3 particular solution]
  Computed value: [2 1 2]
  Expected value: [2, 1, 2]
  check passed

[system 3 homogeneous direction]
  Computed value: [0 0 0]
  Expected value: [0, 0, 0]
  check passed


::::{exercise} Structure of the General Solution of a Nonhomogeneous System
:label: exer-ch6-affine-solution-structure

Consider $x_0+2x_1-x_2+3x_3=2$, $2x_0+4x_1-x_2+8x_3=9$, $-x_0-2x_1+2x_2-x_3=3$.
1. Write the augmented matrix and find its rref.
2. Set the free variables to zero to find a particular solution $\mathbf{x}_p$.
3. Find the general solution $\mathbf{x}_h$ of the corresponding homogeneous system.
4. Write the solution as $\mathbf{x}_p+\mathbf{x}_h$ and explain its geometric relation to the kernel.

**Hint.** Row 2 of the coefficient matrix equals row 1 minus $3$ times row 0; the right-hand side also satisfies $3=9-3\cdot2$.

:::{solution} exer-ch6-affine-solution-structure
:class: dropdown

**Step 1.** The augmented matrix is
$$\left[\begin{array}{rrrr|r}1&2&-1&3&2\\2&4&-1&8&9\\-1&-2&2&-1&3\end{array}\right].$$
After $R_1\leftarrow R_1-2R_0$ and $R_2\leftarrow R_2+R_0$, the last two equations are both $x_2+2x_3=5$. Then performing $R_2\leftarrow R_2-R_1$ and $R_0\leftarrow R_0+R_1$ gives
$$\underline{\operatorname{rref}[\mathbf{A}\mid\mathbf{b}]=\left[\begin{array}{rrrr|r}1&2&0&5&7\\0&0&1&2&5\\0&0&0&0&0\end{array}\right]}.$$
The coefficient matrix and the augmented matrix both have rank $2$, and there are two free variables $x_1,x_3$, so there are infinitely many solutions.

**Step 2.** Setting $x_1=x_3=0$ gives the particular solution
$$\underline{\mathbf{x}_p=(7,0,5,0)^\top}.$$

**Step 3.** The corresponding homogeneous equations are $x_0+2x_1+5x_3=0$ and $x_2+2x_3=0$. Letting $x_1=s$ and $x_3=t$, we get
$$\underline{\mathbf{x}_h=s(-2,1,0,0)^\top+t(-5,0,-2,1)^\top},  s,t\in\mathbb{R}.$$
The two direction vectors are linearly independent and form a basis of $\ker\mathbf{A}$.

**Step 4.** The complete solution of the original system is
$$\underline{\mathbf{x}=\mathbf{x}_p+\mathbf{x}_h=(7-2s-5t,s,5-2t,t)^\top},  s,t\in\mathbb{R}.$$
Since $\mathbf{A}\mathbf{x}_p=\mathbf{b}$ and $\mathbf{A}\mathbf{x}_h=\mathbf{0}$, all of the vectors above are solutions; conversely, the difference between any solution and $\mathbf{x}_p$ belongs to the kernel. Hence the solution set is the affine plane $\mathbf{x}_p+\ker\mathbf{A}$ in $\mathbb{R}^4$.
:::
::::

In [14]:
# ============================================================
print_header("Exercise 6-18 | Structure of the General Solution of a Nonhomogeneous System")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the rref of the augmented matrix and consistency")
A = np.array([[1, 2, -1, 3], [2, 4, -1, 8], [-1, -2, 2, -1]])
b = np.array([2, 9, 3])
augmented = np.column_stack([A, b])
check("rref of the augmented matrix", rref(augmented),
      [[1, 2, 0, 5, 7], [0, 0, 1, 2, 5], [0, 0, 0, 0, 0]])
check("rank(A)", np.linalg.matrix_rank(A), 2)
check("rank([A | b])", np.linalg.matrix_rank(augmented), 2)

# --- Step 2 ---
print_step(2, "Check the particular solution and the homogeneous solution space")
xp = np.array([7, 0, 5, 0])
N = np.array([[-2, -5], [1, 0], [0, -2], [0, 1]])
check("A x_p", A @ xp, b)
check("AN", A @ N, np.zeros((3, 2)))
check("number of kernel basis vectors vs. rank", np.linalg.matrix_rank(N), A.shape[1] - np.linalg.matrix_rank(A))

# --- Step 3 ---
print_step(3, "Substitute the affine general solution x = x_p + x_h")
for s, t in [(0, 0), (1, -2), (-3, 4)]:
    x = xp + N @ np.array([s, t])
    check(f"s={s}, t={t}: general solution", x, [7 - 2*s - 5*t, s, 5 - 2*t, t])
    check("A x", A @ x, b)

  Exercise 6-18 | Structure of the General Solution of a Nonhomogeneous System

▶ Step 1: Check the rref of the augmented matrix and consistency
----------------------------------------

[rref of the augmented matrix]
  Computed value: [[1. 2. 0. 5. 7.]
 [0. 0. 1. 2. 5.]
 [0. 0. 0. 0. 0.]]
  Expected value: [[1, 2, 0, 5, 7], [0, 0, 1, 2, 5], [0, 0, 0, 0, 0]]
  check passed

[rank(A)]
  Computed value: 2
  Expected value: 2
  check passed

[rank([A | b])]
  Computed value: 2
  Expected value: 2
  check passed

▶ Step 2: Check the particular solution and the homogeneous solution space
----------------------------------------

[A x_p]
  Computed value: [2 9 3]
  Expected value: [2 9 3]
  check passed

[AN]
  Computed value: [[0 0]
 [0 0]
 [0 0]]
  Expected value: [[0. 0.]
 [0. 0.]
 [0. 0.]]
  check passed

[number of kernel basis vectors vs. rank]
  Computed value: 2
  Expected value: 2
  check passed

▶ Step 3: Substitute the affine general solution x = x_p + x_h
------------------------

::::{exercise} The Augmented-Rank Criterion
:label: exer-ch6-augmented-rank-alpha

Consider $\mathbf{A}(x,y,z)^\top=\mathbf{b}$, where
$$\mathbf{A}=\begin{pmatrix}1&1&2\\1&2&5\\2&3&7\end{pmatrix},  \mathbf{b}=(2,5,\alpha)^\top.$$
1. Find the rref of the augmented matrix, and determine when there is a solution and what characterizes the case of no solution.
2. Explain why unequal ranks lead to no solution.
3. Prove that $\mathbf{b}\in\operatorname{Col}(\mathbf{A})$ when there is a solution.

**Hint.** Row 2 equals the sum of the first two rows.

:::{solution} exer-ch6-augmented-rank-alpha
:class: dropdown

**Step 1.** Performing $R_1\leftarrow R_1-R_0$ and $R_2\leftarrow R_2-2R_0-R_1$ gives the last two rows $(0,1,3\mid3)$ and $(0,0,0\mid\alpha-7)$. So
$$\underline{\operatorname{rref}[\mathbf{A}\mid\mathbf{b}]=\begin{cases}
\begin{pmatrix}1&0&-1&-1\\0&1&3&3\\0&0&0&0\end{pmatrix},&\alpha=7,\\
\begin{pmatrix}1&0&-1&0\\0&1&3&0\\0&0&0&1\end{pmatrix},&\alpha\ne7.
\end{cases}}$$
There is a solution exactly when $\alpha=7$, and the general solution is $(-1,3,0)^\top+t(1,-3,1)^\top$; in the other cases the last row says $0=1$.

**Step 2.** In the consistent case both ranks are $2$; in the inconsistent case the rank of the augmented matrix is $3$. In general, if a solution exists, then
$$\mathbf{b}=\sum_{j=0}^{2}x_j\operatorname{col}_j(\mathbf{A}),$$
so $\mathbf{b}$ lies in the column space, and adding this column does not increase the rank. Conversely, an increase in rank means that $\mathbf{b}$ is not in the original column space, so it cannot be written as $\mathbf{A}\mathbf{x}$. This also proves part 3. $\square$
:::
::::

In [15]:
# ============================================================
print_header("Exercise 6-19 | The Augmented-Rank Criterion")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the rrefs in the consistent and inconsistent cases")
A=np.array([[1,1,2],[1,2,5],[2,3,7]])
for alpha in [7,0,8]:
    expected=([[1,0,-1,-1],[0,1,3,3],[0,0,0,0]] if alpha==7 else [[1,0,-1,0],[0,1,3,0],[0,0,0,1]])
    check(f"α={alpha}", rref(np.column_stack([A,[2,5,alpha]])), expected)
check("particular solution", A@np.array([-1,3,0]), [2,5,7])
check("homogeneous direction", A@np.array([1,-3,1]), [0,0,0])

  Exercise 6-19 | The Augmented-Rank Criterion

▶ Step 1: Check the rrefs in the consistent and inconsistent cases
----------------------------------------

[α=7]
  Computed value: [[ 1.  0. -1. -1.]
 [ 0.  1.  3.  3.]
 [ 0.  0.  0.  0.]]
  Expected value: [[1, 0, -1, -1], [0, 1, 3, 3], [0, 0, 0, 0]]
  check passed

[α=0]
  Computed value: [[ 1.  0. -1.  0.]
 [ 0.  1.  3.  0.]
 [ 0.  0.  0.  1.]]
  Expected value: [[1, 0, -1, 0], [0, 1, 3, 0], [0, 0, 0, 1]]
  check passed

[α=8]
  Computed value: [[ 1.  0. -1.  0.]
 [ 0.  1.  3.  0.]
 [ 0.  0.  0.  1.]]
  Expected value: [[1, 0, -1, 0], [0, 1, 3, 0], [0, 0, 0, 1]]
  check passed

[particular solution]
  Computed value: [2 5 7]
  Expected value: [2, 5, 7]
  check passed

[homogeneous direction]
  Computed value: [0 0 0]
  Expected value: [0, 0, 0]
  check passed


::::{exercise} Nonhomogeneous System with a Parameter
:label: exer-ch6-parameter-inhomogeneous

Consider the nonhomogeneous system with a real parameter $\lambda$: $x+y+\lambda z=1$, $x+\lambda y+z=1$, $\lambda x+y+z=1$.
1. When is there exactly one solution? Find it.
2. When is there no solution?
3. When are there infinitely many solutions? Find the general solution.

**Hint.** Check the classification of the rank of the coefficient matrix together with the consistency of the constant vector.

:::{solution} exer-ch6-parameter-inhomogeneous
:class: dropdown

**Step 1.** When $\lambda\notin\{1,-2\}$, the coefficient matrix has full rank, and substituting three equal components gives the unique solution
$$\underline{(x,y,z)=\left(\frac1{\lambda+2},\frac1{\lambda+2},\frac1{\lambda+2}\right)}.$$
**Step 2.** When $\lambda=-2$, adding the three equations gives zero on the left but $3$ on the right, so there is $\underline{\text{no solution}}$.

**Step 3.** When $\lambda=1$, all three equations are $x+y+z=1$, so
$$\underline{(x,y,z)^\top=(1,0,0)^\top+s(-1,1,0)^\top+t(-1,0,1)^\top},  s,t\in\mathbb{R}.$$
Its solution set is the translated two-dimensional kernel.
:::
::::

In [16]:
# ============================================================
print_header("Exercise 6-20 | Nonhomogeneous System with a Parameter")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the unique solution for generic parameters")
for lam in [0,2,-3]:
    A=np.array([[1,1,lam],[1,lam,1],[lam,1,1]])
    check(f"λ={lam}: solution", np.linalg.solve(A,np.ones(3)), np.ones(3)/(lam+2))
# --- Step 2 ---
print_step(2, "Check the two special values")
A=np.array([[1,1,-2],[1,-2,1],[-2,1,1]])
check("λ=-2: rank of the augmented matrix", np.linalg.matrix_rank(np.column_stack([A,np.ones(3)])), 3)
A=np.ones((3,3)); N=np.array([[-1,-1],[1,0],[0,1]])
check("λ=1: particular solution", A@np.array([1,0,0]), np.ones(3))
check("λ=1: homogeneous directions", A@N, np.zeros((3,2)))

  Exercise 6-20 | Nonhomogeneous System with a Parameter

▶ Step 1: Check the unique solution for generic parameters
----------------------------------------

[λ=0: solution]
  Computed value: [0.5 0.5 0.5]
  Expected value: [0.5 0.5 0.5]
  check passed

[λ=2: solution]
  Computed value: [0.25 0.25 0.25]
  Expected value: [0.25 0.25 0.25]
  check passed

[λ=-3: solution]
  Computed value: [-1. -1. -1.]
  Expected value: [-1. -1. -1.]
  check passed

▶ Step 2: Check the two special values
----------------------------------------

[λ=-2: rank of the augmented matrix]
  Computed value: 3
  Expected value: 3
  check passed

[λ=1: particular solution]
  Computed value: [1. 1. 1.]
  Expected value: [1. 1. 1.]
  check passed

[λ=1: homogeneous directions]
  Computed value: [[0. 0.]
 [0. 0.]
 [0. 0.]]
  Expected value: [[0. 0.]
 [0. 0.]
 [0. 0.]]
  check passed


### Solution Sets and Affine Translations

::::{exercise} Structure of the General Solution
:label: exer-ch6-particular-plus-kernel

Let $\mathbf{A}$ be an $m\times n$ matrix, and suppose $\mathbf{x}_p$ satisfies $\mathbf{A}\mathbf{x}_p=\mathbf{b}$.
1. Prove that $\mathbf{x}_p+\mathbf{u}$ is a solution for every $\mathbf{u}\in\ker\mathbf{A}$.
2. Prove that every solution $\mathbf{y}$ satisfies $\mathbf{y}-\mathbf{x}_p\in\ker\mathbf{A}$.
3. Explain how a particular solution and a basis of the kernel completely determine the general solution.

**Hint.** Prove the inclusion of the two sets in each direction. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-particular-plus-kernel
:class: dropdown

1. By linearity, $\mathbf{A}(\mathbf{x}_p+\mathbf{u})=\mathbf{A}\mathbf{x}_p+\mathbf{A}\mathbf{u}=\mathbf{b}+\mathbf{0}=\mathbf{b}$.
2. If $\mathbf{A}\mathbf{y}=\mathbf{b}$, then $\mathbf{A}(\mathbf{y}-\mathbf{x}_p)=\mathbf{b}-\mathbf{b}=\mathbf{0}$, so the difference vector lies in the kernel.
3. If a basis of the kernel is $\mathbf{v}_0,\ldots,\mathbf{v}_{d-1}$, then all the solutions, and only these, can be written as $\mathbf{x}=\mathbf{x}_p+\sum_{j=0}^{d-1}t_j\mathbf{v}_j$. When $d=0$, the empty sum is zero and the unique solution is $\mathbf{x}_p$. $\square$
:::
::::

::::{exercise} True or False: Translations
:label: exer-ch6-affine-translation-examples

For a subspace $W\subseteq\mathbb{R}^n$, define $W_{\mathbf{a}}=\{\mathbf{w}+\mathbf{a}:\mathbf{w}\in W\}$. Decide whether each of the following statements is true:
1. In $\mathbb{R}^2$, if $W$ is $y=x$ and $\mathbf{a}=(2,0)^\top$, then $W_{\mathbf{a}}$ is $x-y=2$.
2. In $\mathbb{R}^3$, if $W$ is the line of intersection of $x+y+z=0$ and $x-y-2z=0$ and $\mathbf{a}=(1,2,3)^\top$, then $W_{\mathbf{a}}=\{(1+t,2-2t,3+t)^\top:t\in\mathbb{R}\}$.
3. If $W$ is the plane $x+y+z=0$ and $\mathbf{a}=(1,0,-1)^\top$, then the translate is still the same plane.
4. If $W$ is the plane $x+y+z=0$ and $\mathbf{a}=(-3,0,1)^\top$, then the translate is $\{\mathbf{a}+s(8,-2,-6)^\top+t(-4,1,3)^\top:s,t\in\mathbb{R}\}$.

**Hint.** Translation does not change the direction space, and the two directions of a plane must be linearly independent.

:::{solution} exer-ch6-affine-translation-examples
:class: dropdown

**Step 1.** Statement 1 is true, because $(t,t)^\top+(2,0)^\top=(t+2,t)^\top$, which is exactly the set of all solutions of $x-y=2$. Statement 2 is false: the given direction $(1,-2,1)^\top$ does not satisfy homogeneous equation 1, because $1-(-2)-2=1\ne0$. Solving the two equations gives the correct direction $(1,-3,2)^\top$, so the translate should be $\underline{(1,2,3)^\top+t(1,-3,2)^\top}$.

**Step 2.** Statement 3 is true, because $\mathbf{a}\in W$, so the translate is still $W$. Statement 4 is false, because $(8,-2,-6)^\top=-2(-4,1,3)^\top$, so the two directions span only a line. The actual translated plane is $\underline{x+y+z=-2}$, which can be parametrized, for example, as $(-3,0,1)^\top+s(1,-1,0)^\top+t(1,0,-1)^\top$.

Hence the truth values of the four statements are, in order, $\underline{\text{true, false, true, false}}$.
:::
::::

In [17]:
# ============================================================
print_header("Exercise 6-22 | True or False: Translations")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the wrong and the correct direction in statement 2")
B=np.array([[1,1,1],[1,-1,-2]])
check("residual of the given direction", B@np.array([1,-2,1]), [0,1])
check("residual of the correct direction", B@np.array([1,-3,2]), [0,0])
# --- Step 2 ---
print_step(2, "Check the remaining statements about translations")
check("statement 1: translation constant", np.array([1,-1])@np.array([2,0]), 2)
check("statement 3: translation constant", np.sum([1,0,-1]), 0)
check("statement 4: rank of the directions", np.linalg.matrix_rank(np.array([[8,-4],[-2,1],[-6,3]])), 1)
check("constant of the correct plane", np.sum([-3,0,1]), -2)

  Exercise 6-22 | True or False: Translations

▶ Step 1: Check the wrong and the correct direction in statement 2
----------------------------------------

[residual of the given direction]
  Computed value: [0 1]
  Expected value: [0, 1]
  check passed

[residual of the correct direction]
  Computed value: [0 0]
  Expected value: [0, 0]
  check passed

▶ Step 2: Check the remaining statements about translations
----------------------------------------

[statement 1: translation constant]
  Computed value: 2
  Expected value: 2
  check passed

[statement 3: translation constant]
  Computed value: 0
  Expected value: 0
  check passed

[statement 4: rank of the directions]
  Computed value: 1
  Expected value: 1
  check passed

[constant of the correct plane]
  Computed value: -2
  Expected value: -2
  check passed


::::{exercise} Properties of Translations
:label: exer-ch6-affine-translation-properties

Let $V$ be a linear space, $W\subseteq V$ a subspace, $\mathbf{a},\mathbf{b}\in V$, and $W_{\mathbf{a}}=\{\mathbf{w}+\mathbf{a}:\mathbf{w}\in W\}$. Prove:
1. $\mathbf{a}\in W_{\mathbf{a}}$.
2. If $\mathbf{x}\in W_{\mathbf{a}}$, then $\mathbf{x}-\mathbf{a}\in W$.
3. If $\mathbf{x},\mathbf{y}\in W_{\mathbf{a}}$, then $\mathbf{x}-\mathbf{y}\in W$.
4. If $\mathbf{a}\in W$, then $W_{\mathbf{a}}=W$.
5. If $\mathbf{a}\in W_{\mathbf{b}}$, then $\mathbf{b}\in W_{\mathbf{a}}$.

**Hint.** By the definition, write each member of a set as a vector of the subspace plus the translation vector. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-affine-translation-properties
:class: dropdown

1. Since $\mathbf{0}\in W$, $\mathbf{a}=\mathbf{0}+\mathbf{a}\in W_{\mathbf{a}}$.
2. The definition gives $\mathbf{x}=\mathbf{w}+\mathbf{a}$ with $\mathbf{w}\in W$, so $\mathbf{x}-\mathbf{a}=\mathbf{w}\in W$.
3. Write $\mathbf{x}=\mathbf{u}+\mathbf{a}$ and $\mathbf{y}=\mathbf{v}+\mathbf{a}$; then $\mathbf{x}-\mathbf{y}=\mathbf{u}-\mathbf{v}\in W$.
4. If $\mathbf{a}\in W$, then $\mathbf{w}+\mathbf{a}\in W$, so $W_{\mathbf{a}}\subseteq W$; conversely, every $\mathbf{w}\in W$ can be written as $(\mathbf{w}-\mathbf{a})+\mathbf{a}$, so $W\subseteq W_{\mathbf{a}}$.
5. If $\mathbf{a}=\mathbf{w}+\mathbf{b}$, then $\mathbf{b}=(-\mathbf{w})+\mathbf{a}$ and $-\mathbf{w}\in W$, so $\mathbf{b}\in W_{\mathbf{a}}$. $\square$
:::
::::

::::{exercise} The Solution Set Is Affine
:label: exer-ch6-solution-affine-space

Let $\mathbf{A}$ be an $m\times n$ matrix, and let $\mathbf{x}_p$ be a particular solution of $\mathbf{A}\mathbf{x}=\mathbf{b}$. Prove that the solution set is exactly $\ker(\mathbf{A})_{\mathbf{x}_p}$.

**Hint.** Both inclusions for the translated kernel must be proved. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-solution-affine-space
:class: dropdown

1. If $\mathbf{x}\in\ker(\mathbf{A})_{\mathbf{x}_p}$, then $\mathbf{x}=\mathbf{x}_p+\mathbf{u}$ with $\mathbf{A}\mathbf{u}=\mathbf{0}$, so $\mathbf{A}\mathbf{x}=\mathbf{b}$; hence this translated set is contained in the solution set.
2. If $\mathbf{A}\mathbf{x}=\mathbf{b}$, then $\mathbf{A}(\mathbf{x}-\mathbf{x}_p)=\mathbf{0}$, so $\mathbf{x}-\mathbf{x}_p\in\ker\mathbf{A}$, that is, $\mathbf{x}\in\ker(\mathbf{A})_{\mathbf{x}_p}$.
3. Combining the two directions gives the desired equality. The choice of particular solution does not affect the set: the difference of any two particular solutions lies in the kernel, so they give the same translate. $\square$
:::
::::

### LU and Permutation Matrices

::::{exercise} LU Decompositions of 2×2 Matrices
:label: exer-ch6-two-by-two-lu

Throughout, $\mathbf{L}$ is required to be unit lower triangular and $\mathbf{U}$ upper triangular. For each of the following seven matrices, in order:
$$\begin{pmatrix}2&1\\5&3\end{pmatrix}, \begin{pmatrix}2&0\\10&5\end{pmatrix}, \begin{pmatrix}0&1\\0&-1\end{pmatrix}, \begin{pmatrix}0&1\\1&0\end{pmatrix}, \begin{pmatrix}0&0\\1&0\end{pmatrix}, \begin{pmatrix}1&0\\1&0\end{pmatrix}, \begin{pmatrix}0&0\\0&1\end{pmatrix}.$$
1. Decide whether an LU decomposition exists; if it exists, find all the decompositions and determine uniqueness; if not, explain why.
2. Is the $\mathbf{U}$ of each decomposition a ref?

**Hint.** Compare the four entries of the product.

:::{solution} exer-ch6-two-by-two-lu
:class: dropdown

**Step 1.** Let $\mathbf{L}=\begin{pmatrix}1&0\\\ell&1\end{pmatrix}$ and $\mathbf{U}=\begin{pmatrix}u&v\\0&w\end{pmatrix}$. For $\begin{pmatrix}a&b\\c&d\end{pmatrix}$, the conditions are $u=a,v=b,\ell a=c,w=d-\ell b$. If $a\ne0$, the unique solution is $\ell=c/a$; if $a=0,c\ne0$, there is no solution; if $a=c=0$, $\ell$ is arbitrary.

**Step 2.** The complete classification of each matrix is as follows; in the table, $\ell$ determines $\mathbf{L}$.

| Order | $\ell$ | $\mathbf{U}$ | Uniqueness and ref |
|---|---|---|---|
| 1 | $5/2$ | $\begin{pmatrix}2&1\\0&1/2\end{pmatrix}$ | Unique; a ref |
| 2 | $5$ | $\begin{pmatrix}2&0\\0&5\end{pmatrix}$ | Unique; a ref |
| 3 | Arbitrary | $\begin{pmatrix}0&1\\0&-1-\ell\end{pmatrix}$ | Infinitely many; a ref only when $\ell=-1$ |
| 4 | Does not exist | Does not exist | Because $0\ell=1$ is impossible |
| 5 | Does not exist | Does not exist | Because $0\ell=1$ is impossible |
| 6 | $1$ | $\begin{pmatrix}1&0\\0&0\end{pmatrix}$ | Unique; a ref |
| 7 | Arbitrary | $\begin{pmatrix}0&0\\0&1\end{pmatrix}$ | Infinitely many; none is a ref |

Hence $\underline{\text{an upper triangular matrix is not necessarily a ref}}$; a ref also requires the pivots to move to the right and the zero rows to be at the bottom.
:::
::::

In [18]:
# ============================================================
print_header("Exercise 6-25 | LU Decompositions of 2×2 Matrices")
# ============================================================

# --- Step 1 ---
print_step(1, "Check every matrix for which a decomposition exists")
cases = [(np.array([[2,1],[5,3]]), [2.5]),
         (np.array([[2,0],[10,5]]), [5]),
         (np.array([[0,1],[0,-1]]), [-1,0,2]),
         (np.array([[1,0],[1,0]]), [1]),
         (np.array([[0,0],[0,1]]), [-2,0,3])]
for A, values in cases:
    for ell in values:
        L = np.array([[1,0],[ell,1]])
        U = np.array([[A[0,0],A[0,1]],[0,A[1,1]-ell*A[0,1]]])
        check(f"ℓ={ell}: LU", L@U, A)

  Exercise 6-25 | LU Decompositions of 2×2 Matrices

▶ Step 1: Check every matrix for which a decomposition exists
----------------------------------------

[ℓ=2.5: LU]
  Computed value: [[2. 1.]
 [5. 3.]]
  Expected value: [[2 1]
 [5 3]]
  check passed

[ℓ=5: LU]
  Computed value: [[ 2  0]
 [10  5]]
  Expected value: [[ 2  0]
 [10  5]]
  check passed

[ℓ=-1: LU]
  Computed value: [[ 0  1]
 [ 0 -1]]
  Expected value: [[ 0  1]
 [ 0 -1]]
  check passed

[ℓ=0: LU]
  Computed value: [[ 0  1]
 [ 0 -1]]
  Expected value: [[ 0  1]
 [ 0 -1]]
  check passed

[ℓ=2: LU]
  Computed value: [[ 0  1]
 [ 0 -1]]
  Expected value: [[ 0  1]
 [ 0 -1]]
  check passed

[ℓ=1: LU]
  Computed value: [[1 0]
 [1 0]]
  Expected value: [[1 0]
 [1 0]]
  check passed

[ℓ=-2: LU]
  Computed value: [[0 0]
 [0 1]]
  Expected value: [[0 0]
 [0 1]]
  check passed

[ℓ=0: LU]
  Computed value: [[0 0]
 [0 1]]
  Expected value: [[0 0]
 [0 1]]
  check passed

[ℓ=3: LU]
  Computed value: [[0 0]
 [0 1]]
  Expected value: [[0 0]

::::{exercise} True or False: LU Decompositions
:label: exer-ch6-lu-true-false

For a $2\times2$ square matrix $\mathbf{A}$, decide whether each of the following statements is true or false, and prove your answer (the lower triangular factor of an LU decomposition must be unit lower triangular):
1. If $\det\mathbf{A}\ne0$, there is a unique LU decomposition.
2. If $\det\mathbf{A}\ne0$ and an LU decomposition exists, the decomposition is unique.
3. If $\det\mathbf{A}=0$, an LU decomposition must exist.
4. If the LU decomposition is unique, then $\det\mathbf{A}\ne0$.
5. If there are two or more LU decompositions, then $\det\mathbf{A}=0$.
6. If there is no LU decomposition, then $\det\mathbf{A}\ne0$.

**Hint.** Use the complete classification of the previous exercise to find counterexamples. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-lu-true-false
:class: dropdown

1. False. $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ is invertible but has no LU decomposition, because the lower-left entry of the product must be $\ell\cdot0=0$.
2. True. When a decomposition exists, $\det\mathbf{A}=uw\ne0$, so $u=a\ne0$, and therefore $\ell=c/a,v=b,w=d-cb/a$ are all unique.
3. False. $\begin{pmatrix}0&0\\1&0\end{pmatrix}$ is singular and has no LU decomposition.
4. False. $\begin{pmatrix}1&0\\1&0\end{pmatrix}$ has a unique LU decomposition, yet its determinant is zero.
5. True. This is the contrapositive of statement 2: if the determinant is nonzero and a decomposition exists, there cannot be two decompositions.
6. False. The counterexample to statement 3 also refutes this statement. $\square$
:::
::::

::::{exercise} Conditions on the Leading Pivots
:label: exer-ch6-leading-pivots-lu

1. If $\mathbf{A}=\begin{pmatrix}a_{00}&a_{01}\\a_{10}&a_{11}\end{pmatrix}$ is invertible and $a_{00}\ne0$, prove that the LU decomposition is unique.
2. Explain why the invertibility assumption can be dropped.
3. If $\mathbf{B}=(b_{ij})_{0\le i,j<3}$ satisfies $b_{00}\ne0$, $b_{00}b_{11}-b_{10}b_{01}\ne0$ and is invertible, prove that the LU decomposition is unique.

**Erratum in the original manuscript.** Part 3 mistakenly writes $\mathbf{B}$ as $\mathbf{A}$; here it is written uniformly as $\mathbf{B}$.

**Hint.** First determine the elimination multipliers, then the upper triangular entries. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-leading-pivots-lu
:class: dropdown

1. Comparing entries forces
   $$\mathbf{L}=\begin{pmatrix}1&0\\a_{10}/a_{00}&1\end{pmatrix}, 
   \mathbf{U}=\begin{pmatrix}a_{00}&a_{01}\\0&a_{11}-a_{10}a_{01}/a_{00}\end{pmatrix}.$$
   Their product equals $\mathbf{A}$, and every entry is uniquely determined.
2. The formulas divide only by $a_{00}$ and do not use $\det\mathbf{A}\ne0$, so existence and uniqueness still hold even if the last pivot is zero.
3. Let $\ell_{10}=b_{10}/b_{00}$ and $\ell_{20}=b_{20}/b_{00}$, and set
   $$u_{11}=b_{11}-\ell_{10}b_{01},  u_{12}=b_{12}-\ell_{10}b_{02}, 
   \ell_{21}=\frac{b_{21}-\ell_{20}b_{01}}{u_{11}}, 
   u_{22}=b_{22}-\ell_{20}b_{02}-\ell_{21}u_{12}.$$
   The hypothesis guarantees $u_{11}\ne0$, so
   $$\mathbf{L}=\begin{pmatrix}1&0&0\\\ell_{10}&1&0\\\ell_{20}&\ell_{21}&1\end{pmatrix}, 
   \mathbf{U}=\begin{pmatrix}b_{00}&b_{01}&b_{02}\\0&u_{11}&u_{12}\\0&0&u_{22}\end{pmatrix}$$
   exist and every entry is unique; here we do not even need to assume that $\mathbf{B}$ is invertible. $\square$
:::
::::

::::{exercise} Uniqueness of the Decomposition
:label: exer-ch6-lu-normalization

Let $\mathbf{A}$ be an invertible $n\times n$ matrix, $n\ge1$.
1. If the diagonal of the lower triangular factor is not required to consist of $1$s, is an existing LU decomposition unique?
2. Restoring the unit lower triangular requirement, prove that an existing LU decomposition is unique.

**Hint.** Compare two decompositions to obtain a matrix that is both upper and lower triangular. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-lu-normalization
:class: dropdown

1. Not unique. If $\mathbf{A}=\mathbf{L}\mathbf{U}$, then for any invertible diagonal matrix $\mathbf{D}$ we have $\mathbf{A}=(\mathbf{L}\mathbf{D})(\mathbf{D}^{-1}\mathbf{U})$. Different choices of $\mathbf{D}$ give different factors.
2. If $\mathbf{L}_0\mathbf{U}_0=\mathbf{L}_1\mathbf{U}_1$, then, since $\mathbf{A}$ is invertible, every factor is invertible, and
   $$\mathbf{L}_1^{-1}\mathbf{L}_0=\mathbf{U}_1\mathbf{U}_0^{-1}.$$
   The left side is unit lower triangular and the right side is upper triangular, so the common matrix can only be $\mathbf{I}_n$. Hence $\mathbf{L}_0=\mathbf{L}_1$ and $\mathbf{U}_0=\mathbf{U}_1$. $\square$
:::
::::

::::{exercise} Elimination for a 3×3 Matrix
:label: exer-ch6-three-by-three-lu

Using the elementary row operations $\mathbf{E}_{ij}(c)$ (adding $c$ times row $j$ to row $i$), find the LU decomposition of
$$\mathbf{A}=\begin{pmatrix}2&1&1\\4&5&2\\2&10&7\end{pmatrix}$$
and verify the product.

**Hint.** Enter the elimination multipliers into a unit lower triangular matrix.

:::{solution} exer-ch6-three-by-three-lu
:class: dropdown

**Step 1.** Perform $R_1\leftarrow R_1-2R_0$, $R_2\leftarrow R_2-R_0$, and $R_2\leftarrow R_2-3R_1$ in turn; so
$$\mathbf{U}=\mathbf{E}_{21}(-3)\mathbf{E}_{20}(-1)\mathbf{E}_{10}(-2)\mathbf{A}=\begin{pmatrix}2&1&1\\0&3&0\\0&0&6\end{pmatrix}.$$
**Step 2.** Multiplying in reverse order gives
$$\mathbf{L}=\mathbf{E}_{10}(2)\mathbf{E}_{20}(1)\mathbf{E}_{21}(3)=\begin{pmatrix}1&0&0\\2&1&0\\1&3&1\end{pmatrix},  \underline{\mathbf{L}\mathbf{U}=\mathbf{A}}.$$
:::
::::

In [19]:
# ============================================================
print_header("Exercise 6-29 | Elimination for a 3×3 Matrix")
# ============================================================

# --- Step 1 ---
print_step(1, "Compute and compare the LU factors")
A = np.array([[2,1,1],[4,5,2],[2,10,7]])
L,U = lu_no_pivot(A)
check("L", L, [[1,0,0],[2,1,0],[1,3,1]])
check("U", U, [[2,1,1],[0,3,0],[0,0,6]])
check("LU", L@U, A)

  Exercise 6-29 | Elimination for a 3×3 Matrix

▶ Step 1: Compute and compare the LU factors
----------------------------------------

[L]
  Computed value: [[1. 0. 0.]
 [2. 1. 0.]
 [1. 3. 1.]]
  Expected value: [[1, 0, 0], [2, 1, 0], [1, 3, 1]]
  check passed

[U]
  Computed value: [[2. 1. 1.]
 [0. 3. 0.]
 [0. 0. 6.]]
  Expected value: [[2, 1, 1], [0, 3, 0], [0, 0, 6]]
  check passed

[LU]
  Computed value: [[ 2.  1.  1.]
 [ 4.  5.  2.]
 [ 2. 10.  7.]]
  Expected value: [[ 2  1  1]
 [ 4  5  2]
 [ 2 10  7]]
  check passed


::::{exercise} Decomposing a Rectangular Matrix
:label: exer-ch6-rectangular-lu

Let $\mathbf{C}=\begin{pmatrix}1&2&0&1\\2&4&1&3\\3&6&1&4\end{pmatrix}$.
1. Use row elimination to find a ref $\mathbf{U}$ and a unit lower triangular matrix $\mathbf{L}$ such that $\mathbf{C}=\mathbf{L}\mathbf{U}$.
2. Find the rank, and explain the relation between the nonzero rows of $\mathbf{U}$ and $\operatorname{Row}(\mathbf{C})$.

**Hint.** Record the elimination multipliers $2,3,1$.

:::{solution} exer-ch6-rectangular-lu
:class: dropdown

**Step 1.** Perform $R_1\leftarrow R_1-2R_0$ and $R_2\leftarrow R_2-3R_0$, then $R_2\leftarrow R_2-R_1$, to obtain
$$\mathbf{U}=\begin{pmatrix}1&2&0&1\\0&0&1&1\\0&0&0&0\end{pmatrix}, 
\mathbf{L}=\begin{pmatrix}1&0&0\\2&1&0\\3&1&1\end{pmatrix}.$$
**Step 2.** Multiplying directly gives $\underline{\mathbf{L}\mathbf{U}=\mathbf{C}}$. The two nonzero rows are linearly independent and span the original row space, so $\underline{\operatorname{rank}(\mathbf{C})=2}$, with basis $\{(1,2,0,1),(0,0,1,1)\}$.
:::
::::

In [20]:
# ============================================================
print_header("Exercise 6-30 | Decomposing a Rectangular Matrix")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the LU decomposition and the rank")
C = np.array([[1,2,0,1],[2,4,1,3],[3,6,1,4]])
L = np.array([[1,0,0],[2,1,0],[3,1,1]])
U = np.array([[1,2,0,1],[0,0,1,1],[0,0,0,0]])
check("LU", L@U, C)
check("rank(C)", np.linalg.matrix_rank(C), 2)

  Exercise 6-30 | Decomposing a Rectangular Matrix

▶ Step 1: Check the LU decomposition and the rank
----------------------------------------

[LU]
  Computed value: [[1 2 0 1]
 [2 4 1 3]
 [3 6 1 4]]
  Expected value: [[1 2 0 1]
 [2 4 1 3]
 [3 6 1 4]]
  check passed

[rank(C)]
  Computed value: 2
  Expected value: 2
  check passed


::::{exercise} Permutation Matrices
:label: exer-ch6-permutation-matrices

Let $\bar n=\{0,1,\ldots,n-1\}$ and let $\sigma:\bar n\to\bar n$ be a bijection; define $(\mathbf{P}_\sigma)_{ij}=1$ if $j=\sigma(i)$, and $0$ otherwise.
1. Prove that each row and each column contains exactly one $1$, and prove the converse.
2. Read off the permutations corresponding to the following matrices:
   $$\mathbf{I}_3, \begin{pmatrix}0&1&0\\0&0&1\\1&0&0\end{pmatrix}, \begin{pmatrix}0&1&0&0\\0&0&0&1\\1&0&0&0\\0&0&1&0\end{pmatrix}.$$
3. For an $n\times m$ matrix $\mathbf{A}$, describe the effect of $\mathbf{P}_\sigma\mathbf{A}$.
4. Prove that $\mathbf{P}_\sigma\mathbf{P}_\tau=\mathbf{P}_{\tau\circ\sigma}$.
5. Prove that $\mathbf{P}_\sigma^\top=\mathbf{P}_\sigma^{-1}=\mathbf{P}_{\sigma^{-1}}$.

**Hint.** Compute directly from the definition of the entries; do not switch to another convention for permutation matrices.

:::{solution} exer-ch6-permutation-matrices
:class: dropdown

1. The function has exactly one value at each $i$, so each row contains exactly one $1$; bijectivity guarantees that each value is taken exactly once, so the same holds for each column. Conversely, the column containing the unique $1$ of each row defines $\sigma(i)$; since each column also contains only one $1$, this function is a bijection.
2. Written as lists of images $[\sigma(0),\ldots,\sigma(n-1)]$, the answers are, in order, $[0,1,2]$, $[1,2,0]$, and $[1,3,0,2]$.
3. $\operatorname{row}_i(\mathbf{P}_\sigma\mathbf{A})=\operatorname{row}_{\sigma(i)}(\mathbf{A})$; that is, the new row $i$ is taken from the original row $\sigma(i)$.
4. The entries of the product are $\sum_{k=0}^{n-1}(\mathbf{P}_\sigma)_{ik}(\mathbf{P}_\tau)_{kj}=(\mathbf{P}_\tau)_{\sigma(i),j}$, which equals $1$ exactly when $j=\tau(\sigma(i))$, so the order of composition is as stated.
5. The $(i,j)$ entry of the transpose equals $1$ if and only if $i=\sigma(j)$, which is equivalent to $j=\sigma^{-1}(i)$; hence the transpose equals $\mathbf{P}_{\sigma^{-1}}$. By part 4, multiplying the two in either order gives $\mathbf{I}_n$, so it is also the inverse matrix. $\square$
:::
::::

In [21]:
# ============================================================
print_header("Exercise 6-31 | Permutation Matrices")
# ============================================================

# --- Step 1 ---
print_step(1, "Read off the three permutations")
perms = [[0,1,2],[1,2,0],[1,3,0,2]]
for sigma in perms:
    P = np.eye(len(sigma))[sigma]
    check("column index of the 1 in each row", np.argmax(P,axis=1), sigma)
# --- Step 2 ---
print_step(2, "Check the order of composition with concrete noncommuting permutations")
sigma = np.array([1,3,0,2]); tau = np.array([1,0,2,3])
P = np.eye(4)[sigma]; Q = np.eye(4)[tau]
check("Pσ Pτ", P@Q, np.eye(4)[tau[sigma]])
check("Pσ^T Pσ", P.T@P, np.eye(4))

  Exercise 6-31 | Permutation Matrices

▶ Step 1: Read off the three permutations
----------------------------------------

[column index of the 1 in each row]
  Computed value: [0 1 2]
  Expected value: [0, 1, 2]
  check passed

[column index of the 1 in each row]
  Computed value: [1 2 0]
  Expected value: [1, 2, 0]
  check passed

[column index of the 1 in each row]
  Computed value: [1 3 0 2]
  Expected value: [1, 3, 0, 2]
  check passed

▶ Step 2: Check the order of composition with concrete noncommuting permutations
----------------------------------------

[Pσ Pτ]
  Computed value: [[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]
  Expected value: [[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]
  check passed

[Pσ^T Pσ]
  Computed value: [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
  Expected value: [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
  check passed


::::{exercise} Decomposition with Row Exchanges
:label: exer-ch6-pivoted-lu

For each of the following three matrices $\mathbf{A}$:
$$\begin{pmatrix}0&1&2\\1&2&3\\2&3&4\end{pmatrix}, 
\begin{pmatrix}1&2&3\\2&4&0\\0&2&1\end{pmatrix}, 
\begin{pmatrix}0&1&2\\0&0&1\\1&3&5\end{pmatrix},$$
1. Explain why no LU decomposition is possible without row exchanges.
2. Find a permutation matrix $\mathbf{P}$ such that $\mathbf{P}\mathbf{A}=\mathbf{L}\mathbf{U}$, and write $\mathbf{A}=\mathbf{P}^\top\mathbf{L}\mathbf{U}$.

**Hint.** When there is still a nonzero entry below a zero pivot, rows must be exchanged.

:::{solution} exer-ch6-pivoted-lu
:class: dropdown

**Step 1.** The $(0,0)$ entries of the first and third matrices are zero, but column 0 is nonzero below them; column 0 of any LU product is $u_{00}$ times column 0 of $\mathbf{L}$, so this is impossible. For the second matrix, the lower-right block after the first elimination is $\begin{pmatrix}0&-6\\2&1\end{pmatrix}$, and again the $2$ below cannot be eliminated with a zero pivot.

**Step 2.** In the order of the original exercise, the following factors satisfy $\underline{\mathbf{P}\mathbf{A}=\mathbf{L}\mathbf{U}}$:
$$\begin{aligned}
\mathbf{P}&=\begin{pmatrix}0&1&0\\1&0&0\\0&0&1\end{pmatrix},&
\mathbf{L}&=\begin{pmatrix}1&0&0\\0&1&0\\2&-1&1\end{pmatrix},&
\mathbf{U}&=\begin{pmatrix}1&2&3\\0&1&2\\0&0&0\end{pmatrix};\\
\mathbf{P}&=\begin{pmatrix}1&0&0\\0&0&1\\0&1&0\end{pmatrix},&
\mathbf{L}&=\begin{pmatrix}1&0&0\\0&1&0\\2&0&1\end{pmatrix},&
\mathbf{U}&=\begin{pmatrix}1&2&3\\0&2&1\\0&0&-6\end{pmatrix};\\
\mathbf{P}&=\begin{pmatrix}0&0&1\\1&0&0\\0&1&0\end{pmatrix},&
\mathbf{L}&=\mathbf{I}_3,&
\mathbf{U}&=\begin{pmatrix}1&3&5\\0&1&2\\0&0&1\end{pmatrix}.
\end{aligned}$$
Multiplying each set on the left by $\mathbf{P}^\top$ gives the desired PLU representation; for the third set, take particular care to distinguish $\mathbf{P}$ from $\mathbf{P}^\top$.
:::
::::

In [22]:
# ============================================================
print_header("Exercise 6-32 | Decomposition with Row Exchanges")
# ============================================================

# --- Step 1 ---
print_step(1, "Check PA=LU and A=P^T LU separately")
cases = [([[0,1,2],[1,2,3],[2,3,4]], [1,0,2], [[1,0,0],[0,1,0],[2,-1,1]], [[1,2,3],[0,1,2],[0,0,0]]),
         ([[1,2,3],[2,4,0],[0,2,1]], [0,2,1], [[1,0,0],[0,1,0],[2,0,1]], [[1,2,3],[0,2,1],[0,0,-6]]),
         ([[0,1,2],[0,0,1],[1,3,5]], [2,0,1], np.eye(3), [[1,3,5],[0,1,2],[0,0,1]])]
for A, order, L, U in cases:
    A,L,U = map(np.array,(A,L,U)); P=np.eye(3)[order]
    check("PA", P@A, L@U)
    check("P^T LU", P.T@L@U, A)

  Exercise 6-32 | Decomposition with Row Exchanges

▶ Step 1: Check PA=LU and A=P^T LU separately
----------------------------------------

[PA]
  Computed value: [[1. 2. 3.]
 [0. 1. 2.]
 [2. 3. 4.]]
  Expected value: [[1 2 3]
 [0 1 2]
 [2 3 4]]
  check passed

[P^T LU]
  Computed value: [[0. 1. 2.]
 [1. 2. 3.]
 [2. 3. 4.]]
  Expected value: [[0 1 2]
 [1 2 3]
 [2 3 4]]
  check passed

[PA]
  Computed value: [[1. 2. 3.]
 [0. 2. 1.]
 [2. 4. 0.]]
  Expected value: [[1 2 3]
 [0 2 1]
 [2 4 0]]
  check passed

[P^T LU]
  Computed value: [[1. 2. 3.]
 [2. 4. 0.]
 [0. 2. 1.]]
  Expected value: [[1 2 3]
 [2 4 0]
 [0 2 1]]
  check passed

[PA]
  Computed value: [[1. 3. 5.]
 [0. 1. 2.]
 [0. 0. 1.]]
  Expected value: [[1. 3. 5.]
 [0. 1. 2.]
 [0. 0. 1.]]
  check passed

[P^T LU]
  Computed value: [[0. 1. 2.]
 [0. 0. 1.]
 [1. 3. 5.]]
  Expected value: [[0 1 2]
 [0 0 1]
 [1 3 5]]
  check passed


::::{exercise} Row Exchanges for 2×2 Matrices
:label: exer-ch6-two-by-two-plu

Let $\mathbf{A}=\begin{pmatrix}a&b\\c&d\end{pmatrix}$.
1. If $a\ne0$ and the matrix is invertible, prove that the LU decomposition is unique.
2. Explain why the invertibility assumption is superfluous.
3. Prove that if there is no LU decomposition, then $a=0$. Does the converse hold?
4. In the case where there is no LU decomposition, find a PLU decomposition.

**Hint.** The complete condition for an LU decomposition to exist is $a\ne0$ or $a=c=0$. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-two-by-two-plu
:class: dropdown

1. Comparing the entries of the product forces $\ell=c/a$ and $\mathbf{U}=\begin{pmatrix}a&b\\0&d-cb/a\end{pmatrix}$, so $\mathbf{L}=\begin{pmatrix}1&0\\c/a&1\end{pmatrix}$; multiplying back verifies existence, and uniqueness follows.
2. The formulas above require only $a\ne0$; the last pivot may be zero, so invertibility is not needed.
3. If $a\ne0$ there is an LU decomposition, so by the contrapositive, no LU decomposition implies $a=0$. But $a=0$ is not enough to imply that there is no LU decomposition, for example $\mathbf{0}=\mathbf{I}_2\mathbf{0}$. In fact, when $a=0$ the entry $c$ below must equal $\ell a=0$; if $c=0$, any choice of $\ell$ works with $w=d-\ell b$.
4. There is no LU decomposition exactly when $a=0,c\ne0$. Take $\mathbf{P}=\begin{pmatrix}0&1\\1&0\end{pmatrix}$; then $\mathbf{P}\mathbf{A}=\begin{pmatrix}c&d\\0&b\end{pmatrix}$. Hence we may take $\mathbf{L}=\mathbf{I}_2$ and $\mathbf{U}=\mathbf{P}\mathbf{A}$, and we have $\mathbf{A}=\mathbf{P}^\top\mathbf{L}\mathbf{U}$. $\square$
:::
::::

::::{exercise} Block Decomposition
:label: exer-ch6-block-lu-uniqueness

Let $\mathbf{B}=(b_{ij})_{0\le i,j<3}$ with $b_{00}\ne0$ and $b_{00}b_{11}-b_{10}b_{01}\ne0$. If $\mathbf{B}$ is invertible, use the $2\times2$ result and block multiplication to prove that its LU decomposition is unique.

**Hint.** Separate the first row, the first column, and the lower-right Schur complement. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-block-lu-uniqueness
:class: dropdown

1. Write in blocks $\mathbf{B}=\begin{pmatrix}a&\mathbf{r}\\\mathbf{c}&\mathbf{H}\end{pmatrix}$, where $a=b_{00}\ne0$ and $\mathbf{r}$ is a $1\times2$ block. Then
   $$\mathbf{B}=\begin{pmatrix}1&\mathbf{0}\\\mathbf{c}/a&\mathbf{I}_2\end{pmatrix}
   \begin{pmatrix}a&\mathbf{r}\\\mathbf{0}&\mathbf{S}\end{pmatrix}, 
   \mathbf{S}=\mathbf{H}-\mathbf{c}\mathbf{r}/a.$$
2. The upper-left entry of $\mathbf{S}$ is $(b_{00}b_{11}-b_{10}b_{01})/b_{00}\ne0$, so the $2\times2$ result gives a unique $\mathbf{S}=\mathbf{L}_s\mathbf{U}_s$.
3. Therefore $\mathbf{L}=\begin{pmatrix}1&\mathbf{0}\\\mathbf{c}/a&\mathbf{L}_s\end{pmatrix}$ and $\mathbf{U}=\begin{pmatrix}a&\mathbf{r}\\\mathbf{0}&\mathbf{U}_s\end{pmatrix}$ give the decomposition. Every LU decomposition must have the same factors in the first row and first column, and the remaining factors are determined by the uniqueness for $\mathbf{S}$, so the whole decomposition is unique. The derivation also shows that the invertibility assumption can be dropped. $\square$
:::
::::

::::{exercise} Solving via the Decomposition
:label: exer-ch6-solve-via-plu

If a matrix $\mathbf{A}$ is known to satisfy $\mathbf{P}\mathbf{A}=\mathbf{L}\mathbf{U}$, solving the system $\mathbf{A}\mathbf{x}=\mathbf{b}$ can often be sped up. Suppose
$$\mathbf{P}=\begin{pmatrix}0&1&0\\0&0&1\\1&0&0\end{pmatrix}, 
\mathbf{L}=\begin{pmatrix}1&0&0\\2&1&0\\-1&3&1\end{pmatrix}, 
\mathbf{U}=\begin{pmatrix}2&1&-1\\0&3&2\\0&0&4\end{pmatrix}.$$
1. Solve the nonhomogeneous system $\mathbf{A}\mathbf{x}=\mathbf{b}$, where $\mathbf{b}=(1,2,5)^\top$: in turn find $\mathbf{b}'=\mathbf{P}\mathbf{b}$, solve $\mathbf{L}\mathbf{y}=\mathbf{b}'$, and solve $\mathbf{U}\mathbf{x}=\mathbf{y}$.
2. Explain why these steps solve $\mathbf{A}\mathbf{x}=\mathbf{b}$.

**Hint.** First permute the constant vector, then perform forward substitution and back substitution.

:::{solution} exer-ch6-solve-via-plu
:class: dropdown

**Step 1.** $\mathbf{b}'=(2,5,1)^\top$. Forward substitution gives $y_0=2$, $2y_0+y_1=5$, $-y_0+3y_1+y_2=1$, so $\underline{\mathbf{y}=(2,1,0)^\top}$.

**Step 2.** Back substitution gives $4x_2=0$, $3x_1+2x_2=1$, $2x_0+x_1-x_2=2$, so $\underline{\mathbf{x}=(5/6,1/3,0)^\top}$.

**Step 3.** The two triangular solves give $\mathbf{L}\mathbf{U}\mathbf{x}=\mathbf{P}\mathbf{b}$; substituting $\mathbf{L}\mathbf{U}=\mathbf{P}\mathbf{A}$ and multiplying on the left by $\mathbf{P}^{-1}$ gives the original equation.
:::
::::

In [23]:
# ============================================================
print_header("Exercise 6-35 | Solving via the Decomposition")
# ============================================================

# --- Step 1 ---
print_step(1, "Forward substitution, then back substitution")
P=np.eye(3)[[1,2,0]]; L=np.array([[1,0,0],[2,1,0],[-1,3,1]])
U=np.array([[2,1,-1],[0,3,2],[0,0,4]]); b=np.array([1,2,5])
y=np.linalg.solve(L,P@b); x=np.linalg.solve(U,y)
check("P b", P@b, [2,5,1]); check("y", y, [2,1,0]); check("x", x, [5/6,1/3,0])
# --- Step 2 ---
print_step(2, "Rebuild the coefficient matrix and substitute back")
A=P.T@L@U
check("A x", A@x, b)

  Exercise 6-35 | Solving via the Decomposition

▶ Step 1: Forward substitution, then back substitution
----------------------------------------

[P b]
  Computed value: [2. 5. 1.]
  Expected value: [2, 5, 1]
  check passed

[y]
  Computed value: [ 2.  1. -0.]
  Expected value: [2, 1, 0]
  check passed

[x]
  Computed value: [ 0.8333  0.3333 -0.    ]
  Expected value: [0.8333333333333334, 0.3333333333333333, 0]
  check passed

▶ Step 2: Rebuild the coefficient matrix and substitute back
----------------------------------------

[A x]
  Computed value: [1. 2. 5.]
  Expected value: [1 2 5]
  check passed


### LDU and Symmetric Decompositions

::::{exercise} LDU Decompositions of 2×2 Matrices
:label: exer-ch6-two-by-two-ldu

LDU means that $\mathbf{L}$ is unit lower triangular, $\mathbf{D}$ is diagonal, and $\mathbf{U}$ is unit upper triangular. Classify each of the following matrices as having a unique decomposition, infinitely many decompositions, or no decomposition, and when a decomposition exists, find all the factors:
$$\begin{pmatrix}2&1\\5&3\end{pmatrix}, \begin{pmatrix}2&0\\0&7\end{pmatrix}, \begin{pmatrix}0&1\\1&0\end{pmatrix}, \begin{pmatrix}2&8\\1&4\end{pmatrix},$$
$$\begin{pmatrix}1&0\\0&0\end{pmatrix}, \begin{pmatrix}0&0\\1&-1\end{pmatrix}, \begin{pmatrix}0&1\\0&-1\end{pmatrix}, \begin{pmatrix}0&0\\0&1\end{pmatrix}.$$

**Hint.** Expand the four entries of the product of the three factors.

:::{solution} exer-ch6-two-by-two-ldu
:class: dropdown

**Step 1.** Let $\mathbf{L}=\begin{pmatrix}1&0\\\ell&1\end{pmatrix}$, $\mathbf{D}=\operatorname{diag}(p,q)$, and $\mathbf{U}=\begin{pmatrix}1&u\\0&1\end{pmatrix}$. The product is $\begin{pmatrix}p&pu\\\ell p&\ell pu+q\end{pmatrix}$. If $a\ne0$, the unique solution is $(p,q,\ell,u)=(a,d-cb/a,c/a,b/a)$; if $a=0$, then necessarily $b=c=0$, in which case $p=0,q=d$, while $\ell,u$ are arbitrary.

**Step 2.** In the order of the original exercise:

| Order | $(p,q,\ell,u)$ or reason | Classification |
|---|---|---|
| 1 | $(2,1/2,5/2,1/2)$ | Unique |
| 2 | $(2,7,0,0)$ | Unique |
| 3 | $a=0$ but $b,c\ne0$ | No decomposition |
| 4 | $(2,0,1/2,4)$ | Unique |
| 5 | $(1,0,0,0)$ | Unique |
| 6 | $a=0$ but $c\ne0$ | No decomposition |
| 7 | $a=0$ but $b\ne0$ | No decomposition |
| 8 | $(0,1,\ell,u)$, with $\ell,u\in\mathbb{R}$ arbitrary | Infinitely many |

Hence $\underline{\text{a singular matrix may also have a unique LDU decomposition}}$.
:::
::::

In [24]:
# ============================================================
print_header("Exercise 6-36 | LDU Decompositions of 2×2 Matrices")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the unique decompositions and the family with free parameters")
cases = [([[2,1],[5,3]],(2,.5,2.5,.5)), ([[2,0],[0,7]],(2,7,0,0)),
         ([[2,8],[1,4]],(2,0,.5,4)), ([[1,0],[0,0]],(1,0,0,0))]
for A,(p,q,ell,u) in cases:
    check("LDU", np.array([[1,0],[ell,1]])@np.diag([p,q])@np.array([[1,u],[0,1]]), A)
for ell,u in [(-2,3),(0,0),(4,-1)]:
    check("LDU with free parameters", np.array([[1,0],[ell,1]])@np.diag([0,1])@np.array([[1,u],[0,1]]), [[0,0],[0,1]])

  Exercise 6-36 | LDU Decompositions of 2×2 Matrices

▶ Step 1: Check the unique decompositions and the family with free parameters
----------------------------------------

[LDU]
  Computed value: [[2. 1.]
 [5. 3.]]
  Expected value: [[2, 1], [5, 3]]
  check passed

[LDU]
  Computed value: [[2 0]
 [0 7]]
  Expected value: [[2, 0], [0, 7]]
  check passed

[LDU]
  Computed value: [[2. 8.]
 [1. 4.]]
  Expected value: [[2, 8], [1, 4]]
  check passed

[LDU]
  Computed value: [[1 0]
 [0 0]]
  Expected value: [[1, 0], [0, 0]]
  check passed

[LDU with free parameters]
  Computed value: [[0 0]
 [0 1]]
  Expected value: [[0, 0], [0, 1]]
  check passed

[LDU with free parameters]
  Computed value: [[0 0]
 [0 1]]
  Expected value: [[0, 0], [0, 1]]
  check passed

[LDU with free parameters]
  Computed value: [[0 0]
 [0 1]]
  Expected value: [[0, 0], [0, 1]]
  check passed


::::{exercise} Invertibility and the LDU Decomposition
:label: exer-ch6-invertible-ldu

Prove that if an invertible $3\times3$ matrix has an LU decomposition, then it has a unique LDU decomposition. Does the statement still hold when the invertibility condition is removed?

**Hint.** Extract the diagonal matrix from the upper triangular factor, and check for zero pivots. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-invertible-ldu
:class: dropdown

1. Let $\mathbf{A}=\mathbf{L}\mathbf{V}$, where $\mathbf{V}$ is upper triangular. Invertibility guarantees $v_{ii}\ne0$; taking $\mathbf{D}=\operatorname{diag}(v_{00},v_{11},v_{22})$ and $\mathbf{U}=\mathbf{D}^{-1}\mathbf{V}$ gives an LDU decomposition.
2. Any two LDU decompositions can be regarded as LU decompositions: $\mathbf{L}(\mathbf{D}\mathbf{U})$. By the uniqueness of the LU decomposition of an invertible matrix, $\mathbf{L}$ and $\mathbf{D}\mathbf{U}$ are unique; reading off the diagonal then shows that $\mathbf{D}$ is unique, so $\mathbf{U}$ is unique as well.
3. It does not hold once invertibility is removed. The matrix $\begin{pmatrix}0&1&0\\0&0&0\\0&0&1\end{pmatrix}$ is itself upper triangular, so it has an LU decomposition, but the $(0,0)$ entry of any LDU decomposition forces $d_0=0$, and then $(0,1)=d_0u_{01}=0$, a contradiction; so an LDU decomposition need not even exist. Moreover, $\operatorname{diag}(0,0,1)$ has infinitely many LDU decompositions: for example, take $\mathbf{U}=\mathbf{I}_3$ and $\mathbf{L}=\mathbf{I}_3+t\mathbf{e}_1\mathbf{e}_0^\top$, which works for every $t$. $\square$
:::
::::

::::{exercise} LDU Decomposition with a Parameter
:label: exer-ch6-parameter-ldu

We wish to express $\mathbf{A}=\begin{pmatrix}1&1&1\\1&\lambda^2&0\\1&0&0\end{pmatrix}$ as a unique LDU decomposition.
1. For which real numbers $\lambda$ must rows be exchanged?
2. When no row exchange is needed, find the LDU decomposition.

**Hint.** First set $t=\lambda^2-1$, and note that although $\lambda=0$ gives a singular matrix, the decomposition can still be unique.

:::{solution} exer-ch6-parameter-ldu
:class: dropdown

**Step 1.** Using row 0 to eliminate below it gives $\begin{pmatrix}1&1&1\\0&t&-1\\0&-1&-1\end{pmatrix}$, where $t=\lambda^2-1$. If $t=0$, pivot 1 is zero and the entry below it is $-1$, so rows must be exchanged. Hence a permutation is needed when $\underline{\lambda=\pm1}$; interchanging rows 1 and 2 restores a nonzero pivot.

**Step 2.** When $t\ne0$, the decomposition is
$$\underline{\mathbf{L}=\begin{pmatrix}1&0&0\\1&1&0\\1&-1/t&1\end{pmatrix}, 
\mathbf{D}=\operatorname{diag}(1,t,-\lambda^2/t), 
\mathbf{U}=\begin{pmatrix}1&1&1\\0&1&-1/t\\0&0&1\end{pmatrix}}.$$
The first two diagonal pivots are nonzero, which already determines all the strictly triangular entries uniquely; the last diagonal entry is also determined by the product, so this decomposition is unique even when $\lambda=0$.
:::
::::

In [25]:
# ============================================================
print_header("Exercise 6-38 | LDU Decomposition with a Parameter")
# ============================================================

# --- Step 1 ---
print_step(1, "Check generic values and the singular value λ=0")
for lam in [0,2,-3]:
    t=lam**2-1
    A=np.array([[1,1,1],[1,lam**2,0],[1,0,0]])
    L=np.array([[1,0,0],[1,1,0],[1,-1/t,1]])
    D=np.diag([1,t,-lam**2/t]); U=np.array([[1,1,1],[0,1,-1/t],[0,0,1]])
    check(f"λ={lam}: LDU", L@D@U, A)
# --- Step 2 ---
print_step(2, "Check that elimination works for λ=±1 after a row exchange")
for lam in [-1,1]:
    A=np.array([[1,1,1],[1,lam**2,0],[1,0,0]])
    P=np.eye(3)[[0,2,1]]; L,U=lu_no_pivot(P@A)
    check("PA=LU", P@A, L@U)

  Exercise 6-38 | LDU Decomposition with a Parameter

▶ Step 1: Check generic values and the singular value λ=0
----------------------------------------

[λ=0: LDU]
  Computed value: [[1. 1. 1.]
 [1. 0. 0.]
 [1. 0. 0.]]
  Expected value: [[1 1 1]
 [1 0 0]
 [1 0 0]]
  check passed

[λ=2: LDU]
  Computed value: [[1. 1. 1.]
 [1. 4. 0.]
 [1. 0. 0.]]
  Expected value: [[1 1 1]
 [1 4 0]
 [1 0 0]]
  check passed

[λ=-3: LDU]
  Computed value: [[1. 1. 1.]
 [1. 9. 0.]
 [1. 0. 0.]]
  Expected value: [[1 1 1]
 [1 9 0]
 [1 0 0]]
  check passed

▶ Step 2: Check that elimination works for λ=±1 after a row exchange
----------------------------------------

[PA=LU]
  Computed value: [[1. 1. 1.]
 [1. 0. 0.]
 [1. 1. 0.]]
  Expected value: [[1. 1. 1.]
 [1. 0. 0.]
 [1. 1. 0.]]
  check passed

[PA=LU]
  Computed value: [[1. 1. 1.]
 [1. 0. 0.]
 [1. 1. 0.]]
  Expected value: [[1. 1. 1.]
 [1. 0. 0.]
 [1. 1. 0.]]
  check passed


::::{exercise} Symmetric Decomposition
:label: exer-ch6-symmetric-lu-ldu

What are the LU and LDU decompositions of the symmetric matrix $\mathbf{A}=\begin{pmatrix}1&2&1\\2&7&5\\1&5&10\end{pmatrix}$? Is the upper triangular factor in each equal to $\mathbf{L}^\top$?

**Hint.** Distinguish the upper triangular factor of the LU decomposition from the unit upper triangular factor of the LDU decomposition.

:::{solution} exer-ch6-symmetric-lu-ldu
:class: dropdown

**Step 1.** The elimination multipliers are, in turn, $2,1,1$, giving
$$\mathbf{L}=\begin{pmatrix}1&0&0\\2&1&0\\1&1&1\end{pmatrix}, 
\mathbf{V}=\begin{pmatrix}1&2&1\\0&3&3\\0&0&6\end{pmatrix},  \mathbf{A}=\mathbf{L}\mathbf{V}.$$
**Step 2.** After the diagonal matrix is extracted,
$$\underline{\mathbf{A}=\mathbf{L}\operatorname{diag}(1,3,6)\mathbf{L}^\top}.$$
So in the LU decomposition $\mathbf{V}\ne\mathbf{L}^\top$, but in the LDU decomposition the unit upper triangular factor is $\mathbf{U}=\mathbf{L}^\top$.
:::
::::

In [26]:
# ============================================================
print_header("Exercise 6-39 | Symmetric Decomposition")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the LU and LDL^T decompositions")
A=np.array([[1,2,1],[2,7,5],[1,5,10]])
L,V=lu_no_pivot(A); D=np.diag([1,3,6])
check("L", L, [[1,0,0],[2,1,0],[1,1,1]])
check("V", V, [[1,2,1],[0,3,3],[0,0,6]])
check("LDL^T", L@D@L.T, A)

  Exercise 6-39 | Symmetric Decomposition

▶ Step 1: Check the LU and LDL^T decompositions
----------------------------------------

[L]
  Computed value: [[1. 0. 0.]
 [2. 1. 0.]
 [1. 1. 1.]]
  Expected value: [[1, 0, 0], [2, 1, 0], [1, 1, 1]]
  check passed

[V]
  Computed value: [[1. 2. 1.]
 [0. 3. 3.]
 [0. 0. 6.]]
  Expected value: [[1, 2, 1], [0, 3, 3], [0, 0, 6]]
  check passed

[LDL^T]
  Computed value: [[ 1.  2.  1.]
 [ 2.  7.  5.]
 [ 1.  5. 10.]]
  Expected value: [[ 1  2  1]
 [ 2  7  5]
 [ 1  5 10]]
  check passed


::::{exercise} A Degenerate Symmetric Form
:label: exer-ch6-singular-ldlt-family

Let $\mathbf{A}=\begin{pmatrix}0&0\\0&3\end{pmatrix}$. Explain how it can be written as $\mathbf{L}\mathbf{D}\mathbf{L}^\top$, where $\mathbf{L}$ is unit lower triangular and $\mathbf{D}$ is diagonal. Is this representation unique?

**Hint.** Simply let the lower triangular entry be an arbitrary parameter.

:::{solution} exer-ch6-singular-ldlt-family
:class: dropdown

**Step 1.** Let $\mathbf{L}=\begin{pmatrix}1&0\\t&1\end{pmatrix}$. The first diagonal entry of the product forces $d_0=0$, and the last entry then forces $d_1=3$.

**Step 2.** For every $t\in\mathbb{R}$,
$$\underline{\begin{pmatrix}1&0\\t&1\end{pmatrix}\operatorname{diag}(0,3)\begin{pmatrix}1&t\\0&1\end{pmatrix}=\mathbf{A}}.$$
So $\mathbf{D}$ is fixed, but $\mathbf{L}$ can vary freely, and there are infinitely many decompositions.
:::
::::

In [27]:
# ============================================================
print_header("Exercise 6-40 | A Degenerate Symmetric Form")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the decomposition for different t")
for t in [-2,0,3]:
    L=np.array([[1,0],[t,1]])
    check(f"t={t}: LDL^T", L@np.diag([0,3])@L.T, [[0,0],[0,3]])

  Exercise 6-40 | A Degenerate Symmetric Form

▶ Step 1: Check the decomposition for different t
----------------------------------------

[t=-2: LDL^T]
  Computed value: [[0 0]
 [0 3]]
  Expected value: [[0, 0], [0, 3]]
  check passed

[t=0: LDL^T]
  Computed value: [[0 0]
 [0 3]]
  Expected value: [[0, 0], [0, 3]]
  check passed

[t=3: LDL^T]
  Computed value: [[0 0]
 [0 3]]
  Expected value: [[0, 0], [0, 3]]
  check passed


::::{exercise} Classifying Symmetric Decompositions
:label: exer-ch6-two-by-two-ldlt

For a real symmetric matrix $\mathbf{A}=\begin{pmatrix}a&b\\b&c\end{pmatrix}$, classify its LDL$^\top$ decompositions with unit lower triangular $\mathbf{L}$ and diagonal $\mathbf{D}$: when are they unique, infinitely many, or nonexistent? Explain the relation to the rank.

**Hint.** Compare $a=d_0$, $b=\ell d_0$, $c=\ell^2d_0+d_1$. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-two-by-two-ldlt
:class: dropdown

1. If $a\ne0$, the unique decomposition is $\mathbf{L}=\begin{pmatrix}1&0\\b/a&1\end{pmatrix}$, $\mathbf{D}=\operatorname{diag}(a,c-b^2/a)$. If $ac-b^2\ne0$ the rank is $2$; otherwise the rank is $1$.
2. If $a=0,b\ne0$, the condition $b=\ell a$ is impossible, so there is no decomposition; in this case the determinant is $-b^2\ne0$, and the rank is $2$.
3. If $a=b=0$, then $\mathbf{D}=\operatorname{diag}(0,c)$ while $\ell$ is arbitrary, so there are infinitely many decompositions. The rank is $1$ when $c\ne0$ and $0$ when $c=0$.
4. Hence rank $2$ may give a unique decomposition or none, rank $1$ may give a unique decomposition or infinitely many, and rank $0$ gives infinitely many; the rank alone does not determine the type of decomposition. $\square$
:::
::::

::::{exercise} Deriving the Symmetric Form
:label: exer-ch6-symmetric-lu-implies-ldlt

If a real symmetric invertible matrix $\mathbf{A}$ has an LU decomposition, prove that it can be written as $\mathbf{A}=\mathbf{L}\mathbf{D}\mathbf{L}^\top$, where $\mathbf{L}$ is unit lower triangular and $\mathbf{D}$ is diagonal. Does existence still hold when the invertibility condition is removed?

**Hint.** A matrix that is both symmetric and upper triangular must be diagonal. (This is a derivation/proof exercise; no Python verification is attached.)

:::{solution} exer-ch6-symmetric-lu-implies-ldlt
:class: dropdown

1. Let $\mathbf{A}=\mathbf{L}\mathbf{U}$. A unit lower triangular matrix $\mathbf{L}$ is always invertible, whether or not $\mathbf{A}$ is invertible. Let
   $$\mathbf{D}=\mathbf{L}^{-1}\mathbf{A}\mathbf{L}^{-\top}=\mathbf{U}\mathbf{L}^{-\top}.$$
2. The right side is a product of upper triangular matrices, so $\mathbf{D}$ is upper triangular; on the left side, $\mathbf{A}^\top=\mathbf{A}$ shows that $\mathbf{D}^\top=\mathbf{D}$, so it must be a diagonal matrix. Multiplying back gives $\mathbf{A}=\mathbf{L}\mathbf{D}\mathbf{L}^\top$.
3. The proof never uses the invertibility of $\mathbf{A}$, so once invertibility is removed, existence still holds as long as $\mathbf{A}$ is symmetric and has an LU decomposition. Uniqueness, however, need not hold: for example, $\operatorname{diag}(0,3)$ has the family with a free parameter given in the previous exercise. $\square$
:::
::::

### Augmented Matrices and Inversion

::::{exercise} Gauss–Jordan Inversion
:label: exer-ch6-gauss-jordan-inverse

Let $\mathbf{A}=\begin{pmatrix}1&0&0&1\\0&1&-1&2\\2&-1&2&0\\1&0&2&2\end{pmatrix}$.
1. Form $[\mathbf{A}\mid\mathbf{I}_4]$.
2. Use Gauss–Jordan elimination to reduce the left half to $\mathbf{I}_4$.
3. Write down the matrix $\mathbf{B}$ on the right.
4. Verify that $\mathbf{A}\mathbf{B}=\mathbf{B}\mathbf{A}=\mathbf{I}_4$.

**Hint.** Apply the same row operations simultaneously to the left and right halves of the augmented matrix, eliminating downward first and then upward.

:::{solution} exer-ch6-gauss-jordan-inverse
:class: dropdown

**Step 1.** The augmented matrix is
$$\left[\begin{array}{rrrr|rrrr}1&0&0&1&1&0&0&0\\0&1&-1&2&0&1&0&0\\2&-1&2&0&0&0&1&0\\1&0&2&2&0&0&0&1\end{array}\right].$$
Performing $R_2\leftarrow R_2-2R_0$, $R_2\leftarrow R_2+R_1$, $R_3\leftarrow R_3-R_0$, and $R_3\leftarrow R_3-2R_2$ in turn gives
$$\left[\begin{array}{rrrr|rrrr}1&0&0&1&1&0&0&0\\0&1&-1&2&0&1&0&0\\0&0&1&0&-2&1&1&0\\0&0&0&1&3&-2&-2&1\end{array}\right].$$
The left side is an upper triangular matrix whose diagonal entries are all $1$, and the row-addition operations above do not change the determinant, so $\det\mathbf{A}=1$ and the matrix is invertible.

**Step 2.** Then perform $R_0\leftarrow R_0-R_3$, $R_1\leftarrow R_1-2R_3$, and $R_1\leftarrow R_1+R_2$ in turn to obtain
$$\left[\begin{array}{rrrr|rrrr}1&0&0&0&-2&2&2&-1\\0&1&0&0&-8&6&5&-2\\0&0&1&0&-2&1&1&0\\0&0&0&1&3&-2&-2&1\end{array}\right].$$
Hence the matrix on the right is
$$\underline{\mathbf{B}=\mathbf{A}^{-1}=\begin{pmatrix}-2&2&2&-1\\-8&6&5&-2\\-2&1&1&0\\3&-2&-2&1\end{pmatrix}}.$$

**Step 3.** Multiplying directly gives
$$\mathbf{A}\mathbf{B}=\mathbf{B}\mathbf{A}=\begin{pmatrix}1&0&0&0\\0&1&0&0\\0&0&1&0\\0&0&0&1\end{pmatrix}=\mathbf{I}_4,$$
which verifies that the matrix above is indeed the inverse matrix.
:::
::::

In [28]:
# ============================================================
print_header("Exercise 6-43 | Gauss–Jordan Inversion")
# ============================================================

# --- Step 1 ---
print_step(1, "Form the augmented matrix and eliminate downward")
A = np.array([[1, 0, 0, 1], [0, 1, -1, 2], [2, -1, 2, 0], [1, 0, 2, 2]])
G = np.column_stack([A, np.eye(4)])
print("[A | I_4] =\n", G)
G[2] -= 2 * G[0]
G[2] += G[1]
G[3] -= G[0]
G[3] -= 2 * G[2]
check("augmented matrix after downward elimination", G,
      [[1, 0, 0, 1, 1, 0, 0, 0], [0, 1, -1, 2, 0, 1, 0, 0],
       [0, 0, 1, 0, -2, 1, 1, 0], [0, 0, 0, 1, 3, -2, -2, 1]])
check("det(A)", np.linalg.det(A), 1)
check("rank(A)", np.linalg.matrix_rank(A), 4)

# --- Step 2 ---
print_step(2, "Eliminate upward and read off the inverse matrix")
G[0] -= G[3]
G[1] -= 2 * G[3]
G[1] += G[2]
B = G[:, 4:]
expected_inverse = np.array([[-2, 2, 2, -1], [-8, 6, 5, -2],
                             [-2, 1, 1, 0], [3, -2, -2, 1]])
check("left half", G[:, :4], np.eye(4))
check("A^{-1}", B, expected_inverse)

# --- Step 3 ---
print_step(3, "Check by multiplying on both sides")
check("AB", A @ B, np.eye(4))
check("BA", B @ A, np.eye(4))

  Exercise 6-43 | Gauss–Jordan Inversion

▶ Step 1: Form the augmented matrix and eliminate downward
----------------------------------------
[A | I_4] =
 [[ 1.  0.  0.  1.  1.  0.  0.  0.]
 [ 0.  1. -1.  2.  0.  1.  0.  0.]
 [ 2. -1.  2.  0.  0.  0.  1.  0.]
 [ 1.  0.  2.  2.  0.  0.  0.  1.]]

[augmented matrix after downward elimination]
  Computed value: [[ 1.  0.  0.  1.  1.  0.  0.  0.]
 [ 0.  1. -1.  2.  0.  1.  0.  0.]
 [ 0.  0.  1.  0. -2.  1.  1.  0.]
 [ 0.  0.  0.  1.  3. -2. -2.  1.]]
  Expected value: [[1, 0, 0, 1, 1, 0, 0, 0], [0, 1, -1, 2, 0, 1, 0, 0], [0, 0, 1, 0, -2, 1, 1, 0], [0, 0, 0, 1, 3, -2, -2, 1]]
  check passed

[det(A)]
  Computed value: 0.9999999999999998
  Expected value: 1
  check passed

[rank(A)]
  Computed value: 4
  Expected value: 4
  check passed

▶ Step 2: Eliminate upward and read off the inverse matrix
----------------------------------------

[left half]
  Computed value: [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
  Expected valu

::::{exercise} Inversion with a Parameter
:label: exer-ch6-parameter-inverse

Let $\mathbf{B}_a=\begin{pmatrix}1&1&0\\1&2&1\\0&2&a\end{pmatrix}$.
1. Perform elimination on $[\mathbf{B}_a\mid\mathbf{I}_3]$. When does a pivot become zero in a way that no row exchange can remedy? In that case, use the rank to explain whether the inverse matrix exists.
2. When $a=3$, find the inverse matrix by Gauss–Jordan elimination.

**Hint.** The first two pivots are fixed at $1$, and the last one is $a-2$.

:::{solution} exer-ch6-parameter-inverse
:class: dropdown

**Step 1.** Performing $R_1\leftarrow R_1-R_0$ and $R_2\leftarrow R_2-2R_1$ gives
$$\left[\begin{array}{rrr|rrr}1&1&0&1&0&0\\0&1&1&-1&1&0\\0&0&a-2&2&-2&1\end{array}\right].$$
When $\underline{a=2}$, the last row of the left side is entirely zero and only two pivots remain, so the rank is $2$ and the inverse matrix does not exist; for other values the rank is $3$.

**Step 2.** When $a=3$, the last pivot is $1$; then performing $R_1\leftarrow R_1-R_2$ and $R_0\leftarrow R_0-R_1$ gives
$$\underline{\mathbf{B}_3^{-1}=\begin{pmatrix}4&-3&1\\-3&3&-1\\2&-2&1\end{pmatrix}}.$$
:::
::::

In [29]:
# ============================================================
print_header("Exercise 6-44 | Inversion with a Parameter")
# ============================================================

# --- Step 1 ---
print_step(1, "Check the degenerate parameter")
check("rank for a=2", np.linalg.matrix_rank([[1,1,0],[1,2,1],[0,2,2]]), 2)
# --- Step 2 ---
print_step(2, "Check the inverse matrix on both sides")
B=np.array([[1,1,0],[1,2,1],[0,2,3]])
C=np.array([[4,-3,1],[-3,3,-1],[2,-2,1]])
check("B C", B@C, np.eye(3)); check("C B", C@B, np.eye(3))

  Exercise 6-44 | Inversion with a Parameter

▶ Step 1: Check the degenerate parameter
----------------------------------------

[rank for a=2]
  Computed value: 2
  Expected value: 2
  check passed

▶ Step 2: Check the inverse matrix on both sides
----------------------------------------

[B C]
  Computed value: [[1 0 0]
 [0 1 0]
 [0 0 1]]
  Expected value: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
  check passed

[C B]
  Computed value: [[1 0 0]
 [0 1 0]
 [0 0 1]]
  Expected value: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
  check passed


---

✓ **All exercises completed.**